# Hare Escape Optimization (HEO): Sequential vs OpenMP vs CUDA performance study

This notebook implements the **Hare Escape Optimization (HEO)** metaheuristic (Alsamee & Ramezani, *Scientific Reports* 15:26405, 2025) three times (sequential CPU, OpenMP CPU, CUDA GPU) and compares them on the same benchmark functions with matched seeds. It is designed for Google Colab (Runtime → Change runtime type → T4 GPU) but also runs CPU-only.

## What is HEO?

HEO is a population-based, single-objective minimiser inspired by a hare fleeing a wolf. Each of the $N$ hares keeps a position $X_j$ and a velocity $V_j$ (both $D$-dimensional). In iteration $t = 1,\dots,T$ every hare proposes one candidate (paper Eq. 4):

$$X^{t+1}_{j,i} = X^{t}_{j,i} + \frac{Maxit}{t}\,V^{t}_{j,i} + L\,\big(X^{t}_{j,i} + V^{t}_{j,i}\big)\,r ,\qquad L\sim\text{Lévy flight},\; r\sim U(0,1)$$

The velocity is then reduced (the paper states a velocity-reduction coefficient of 1 and damping of 0.95). The $N$ parents and $N$ candidates are **merged, sorted by fitness, and the best $N$ survive** (elitist selection); the first element of the sorted population is the best solution. Several details of the pseudocode are under-specified or contain typos. They are listed, with the choice made for each, in the header comment of `heo_common.h` (Section 2) rather than being changed silently.

## The three implementations

| Version | Source | Parallelism |
|---|---|---|
| Sequential (reference) | `heo_seq.cpp` | single thread; defines the behaviour the other two must reproduce |
| OpenMP | `heo_omp.cpp` | hares are updated and evaluated in parallel on CPU threads; thread count is configurable |
| CUDA | `heo_cuda.cu` | initialisation, update, evaluation, selection and gather run on the GPU; data stay in device memory for the whole run |

All three include the same `heo_common.h` (counter-based RNG, Lévy step, update rule, benchmark functions) and the same command-line driver `heo_main.h`. Because the RNG is stateless (a function of seed, iteration, hare, dimension and stream), the random numbers do not depend on thread scheduling. CPU versions are expected to agree exactly; the GPU may differ at rounding level because CUDA's `sin`, `cos`, `log`, `exp` and `pow` are not bit-identical to the host libm. The notebook measures these differences instead of assuming they are zero.

## Objectives

1. **Correctness:** check that OpenMP and CUDA reproduce the sequential reference for identical seeds (numerical parity).
2. **Performance:** compare runtime, speedup, parallel efficiency and overhead across population size, dimension and iteration count.
3. **Timing transparency:** report *end-to-end* time (what a user waits for, including GPU allocation and transfers) separately from *compute* time (CUDA events for the GPU kernels).
4. **Solution quality:** report best, mean, worst, median, standard deviation and IQR of the final fitness over repeated independent runs, with non-parametric tests (Wilcoxon, Friedman) where they apply.
5. **Scaling:** show where parallel versions begin to pay off (population, dimension and OpenMP-thread sweeps).

## Scope and caveats

* CEC 2015 needs the official shift/rotation data, which is not bundled, so the benchmark set is the paper's F16–F19 and F24–F30 plus Ackley, Griewank and Rosenbrock. The paper's printed formulas for Easom, Beale and Schaffer N.2 contain typos; the standard forms are used.
* The update term $L\,(X+V)\,r$ scales with the position itself, which favours optima at the origin. The `--shift` runs move the optimum away from the origin to expose this.
* This notebook evaluates **parallel performance and implementation parity**. It does not validate the paper's quality claims against its 29 competitors.
* Nothing here is pre-computed: every table and figure is produced by running the cells. If a stage cannot run (for example no GPU), the notebook says so and skips it rather than filling in values.


In [ ]:
# Cell 1: environment inventory. Reports Python, C++ compiler/OpenMP, CPU, RAM, CUDA toolkit and GPU.
# Missing GPU/CUDA is reported (not fatal here); the GPU requirement is enforced in Section 1's runtime
# check below. A missing C++ compiler IS fatal, because even the CPU versions cannot be built without it.
import os, sys, io, re, json, shutil, subprocess, time, platform
from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _cmd(cmd, timeout=30):
    """Run a command, never raise. Returns (returncode, stdout+stderr stripped); rc=None if it could not start."""
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout + p.stderr).strip()
    except FileNotFoundError:
        return None, f"{cmd[0]}: not found"
    except subprocess.TimeoutExpired:
        return None, f"{' '.join(cmd)}: timed out after {timeout}s"

ENV = {"python": sys.version.split()[0], "platform": platform.platform(), "cpu_count": os.cpu_count()}

# ---- CPU model and RAM (Linux; falls back gracefully elsewhere)
ENV["cpu_model"] = platform.processor() or "unknown"
try:
    for line in open("/proc/cpuinfo"):
        if line.lower().startswith("model name"):
            ENV["cpu_model"] = line.split(":", 1)[1].strip(); break
    for line in open("/proc/meminfo"):
        if line.startswith("MemTotal"):
            ENV["ram_gb"] = round(int(line.split()[1]) / 1024 / 1024, 1); break
except OSError:
    pass

# ---- C++ compiler and OpenMP support
gxx = shutil.which("g++")
if gxx is None:
    raise RuntimeError("g++ was not found. The sequential and OpenMP versions need a C++17 compiler "
                       "(on Debian/Ubuntu: `apt-get install -y g++`).")
ENV["gxx"] = _cmd(["g++", "--version"])[1].splitlines()[0]
rc, out = _cmd(["bash", "-c", "echo | g++ -fopenmp -dM -E -x c++ - | grep -w _OPENMP"])
ENV["openmp_macro"] = out.split()[-1] if rc == 0 and out else None
if ENV["openmp_macro"] is None:
    raise RuntimeError("g++ does not accept -fopenmp (libgomp missing?). Install it, e.g. `apt-get install -y libgomp1`.")

# ---- CUDA toolkit and GPU
nvcc, smi = shutil.which("nvcc"), shutil.which("nvidia-smi")
ENV["nvcc"] = None
if nvcc:
    m = re.search(r"release ([\d.]+)", _cmd(["nvcc", "--version"])[1])
    ENV["nvcc"] = m.group(1) if m else "installed (version not parsed)"
ENV["gpu"] = None
if smi:
    fields = "name,memory.total,memory.free,driver_version,compute_cap"
    rc, out = _cmd(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"])
    if rc != 0:      # older drivers do not know compute_cap
        fields = "name,memory.total,memory.free,driver_version"
        rc, out = _cmd(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"])
    if rc == 0 and out:
        vals = [v.strip() for v in out.splitlines()[0].split(",")]
        ENV["gpu"] = dict(zip(fields.split(","), vals))
    else:
        ENV["gpu_error"] = out

print("=== Environment ===")
for k in ["python", "platform", "cpu_model", "cpu_count", "ram_gb", "gxx", "openmp_macro", "nvcc"]:
    print(f"{k:13s}: {ENV.get(k)}")
if ENV["gpu"]:
    print("gpu          :", ENV["gpu"])

# ---- actionable diagnostics (warnings only; Section 1 decides whether to stop)
if smi is None:
    print("\nWARNING: nvidia-smi not found, so no NVIDIA GPU is visible. In Colab choose "
          "Runtime > Change runtime type > T4 GPU, then restart. CPU-only runs remain possible.")
elif ENV["gpu"] is None:
    print("\nWARNING: nvidia-smi exists but could not query a GPU:", ENV.get("gpu_error", "unknown error"),
          "\n         (driver not loaded or no device attached to this runtime).")
if ENV["gpu"] and nvcc is None:
    print("\nWARNING: a GPU is present but nvcc is missing, so the CUDA version cannot be compiled. "
          "Use a runtime with the CUDA toolkit (Colab GPU runtimes normally provide it).")


## 1. Select and validate the GPU runtime

The next cell decides which back-ends can run. It checks that an NVIDIA GPU is visible, that `nvcc` can actually compile and launch a trivial kernel (not merely that it is on the `PATH`), and how much GPU memory is free. With `REQUIRE_GPU = True` the notebook stops with an explanatory message if any check fails; with `False` it continues with the sequential and OpenMP versions only and every later CUDA step is skipped.


In [ ]:
# Cell 3: runtime configuration and GPU validation.
import os, shutil, subprocess, time, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REQUIRE_GPU = True      # set False to run only the sequential + OpenMP versions

HAS_GPU, GPU_INFO, CUDA_ARCH, GPU_FREE_MB = False, None, None, None
gpu_problem = None      # human-readable reason if the CUDA version cannot be used

def _try(cmd, timeout=120):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout + p.stderr).strip()
    except FileNotFoundError:
        return None, f"{cmd[0]} not found"
    except subprocess.TimeoutExpired:
        return None, f"{cmd[0]} timed out after {timeout}s"

if ENV.get("gpu") is None:
    gpu_problem = ("No NVIDIA GPU is visible (nvidia-smi missing or failed). In Colab: Runtime > Change runtime "
                   "type > T4 GPU, then restart the runtime.")
elif ENV.get("nvcc") is None:
    gpu_problem = "A GPU is visible but nvcc is not installed, so heo_cuda.cu cannot be compiled."
else:
    # Compile AND launch a trivial kernel: proves toolchain, architecture flag, driver and device all work.
    GPU_INFO = ENV["gpu"]
    cc = GPU_INFO.get("compute_cap")
    CUDA_ARCH = f"sm_{cc.replace('.', '')}" if cc and cc.replace(".", "").isdigit() else None
    open("cuda_probe.cu", "w").write(
        "#include <cstdio>\n#include <cuda_runtime.h>\n"
        "__global__ void k(int* p){ p[threadIdx.x] = threadIdx.x; }\n"
        "int main(){ int n=0; if(cudaGetDeviceCount(&n)!=cudaSuccess||n==0){puts(\"no device\");return 2;}\n"
        " int* d; cudaMalloc(&d,32*sizeof(int)); k<<<1,32>>>(d); cudaError_t e=cudaDeviceSynchronize();\n"
        " if(e!=cudaSuccess){printf(\"launch failed: %s\\n\",cudaGetErrorString(e));return 3;}\n"
        " cudaFree(d); printf(\"ok %d device(s)\\n\", n); return 0; }\n")
    cmd = ["nvcc", "-O1", "cuda_probe.cu", "-o", "cuda_probe"] + ([f"-arch={CUDA_ARCH}"] if CUDA_ARCH else [])
    rc, out = _try(cmd)
    if rc != 0:
        gpu_problem = f"nvcc could not compile a trivial kernel ({' '.join(cmd)}):\n{out}"
    else:
        rc, out = _try(["./cuda_probe"])
        if rc != 0:
            gpu_problem = f"A trivial CUDA kernel compiled but failed to run: {out}"
        else:
            HAS_GPU = True
            try:
                GPU_FREE_MB = float(GPU_INFO["memory.free"])
            except (KeyError, ValueError):
                GPU_FREE_MB = None

def cuda_bytes(pop, dim, real_bytes=8):
    """Approximate device memory (bytes) used by heo_cuda for one run: 4 position/velocity buffers of
    (2N x D) reals, fitness + index buffers of length 2N, the two N x D term buffers used by HEO_CUDA_EVAL=terms,
    and ~25% headroom for the CUB sort workspace."""
    M = 2 * pop
    return int(1.25 * (4 * M * dim * real_bytes + 2 * pop * dim * 8 + M * (2 * 8 + 2 * 4)))

def fits_on_gpu(pop, dim):
    """True if a (pop, dim) run is expected to fit in free GPU memory (True if free memory is unknown)."""
    return GPU_FREE_MB is None or cuda_bytes(pop, dim) < 0.9 * GPU_FREE_MB * 1024 * 1024

print("=== Runtime configuration ===")
print("REQUIRE_GPU      :", REQUIRE_GPU)
print("HAS_GPU          :", HAS_GPU)
if HAS_GPU:
    print("GPU              :", GPU_INFO.get("name"), "| driver", GPU_INFO.get("driver_version"),
          "| compute capability", GPU_INFO.get("compute_cap", "unknown"))
    print("GPU memory       : total", GPU_INFO.get("memory.total"), "MiB | free", GPU_INFO.get("memory.free"), "MiB")
    print("nvcc / arch flag :", ENV["nvcc"], "/", CUDA_ARCH or "(none; nvcc default)")
    print("largest sweep point fits (N=16384, D=1000):", fits_on_gpu(16384, 1000))
else:
    print("CUDA unavailable :", gpu_problem)
if not HAS_GPU and REQUIRE_GPU:
    raise RuntimeError(gpu_problem + "\nTo continue without the GPU, set REQUIRE_GPU = False in this cell and re-run it.")
if not HAS_GPU:
    print("\nContinuing in CPU-ONLY mode: only the sequential and OpenMP versions will be built and benchmarked.")
print("CPU threads visible:", os.cpu_count())


## 2. Write the source files

All sources are embedded in the notebook (via `%%writefile`), so no uploads are needed. Five files are generated; the three back-ends share everything except the parallel strategy.

| File | Role | Contains |
|---|---|---|
| `heo_common.h` | **Single source of truth for the algorithm** (compiled by all three back-ends) | paper-to-code map and documented ambiguities, counter-based RNG, Lévy step, HEO update rule (Eq. 4), 14 benchmark functions, `Params`/`RunInfo`, the merge-selection helper |
| `heo_main.h` | **Common command-line driver** | argument parsing and validation, untimed warm-up, repeated independent runs, wall-clock timing, CSV output, convergence-history export, `--selftest` |
| `heo_seq.cpp` | **Reference implementation** | single-threaded HEO; defines the behaviour the others must reproduce |
| `heo_omp.cpp` | OpenMP back-end | parallel initialisation, update + evaluation, candidate sort, selection + gather |
| `heo_cuda.cu` | CUDA back-end | the same steps as GPU kernels (CUB radix sort of candidates, merge-path selection) with all intermediate data resident on the device |

**Back-end contract.** A back-end defines `VERSION_NAME`, `heo_backend_init(threads)` and `heo_run(Params, seed, RunInfo*)`, then includes `heo_main.h`. `heo_run` returns the best fitness found. Through `RunInfo` it also reports `compute_ms` (time of the algorithm loop only: CUDA events on the GPU, a clock around the loop on the CPU) and, if requested, the best-so-far value after every iteration.

**One HEO iteration** (identical in all back-ends):

```
for each hare j in parallel : candidate_j = Eq.4(parent_j) ; fitness_j = f(candidate_j)
sort the N candidates       : by (fitness, index)           -- strict total order => deterministic
merge parents + candidates  : keep the best N, parents win ties (elitist selection)
gather the survivors        : positions, velocities, fitness; best = first survivor
```

**Command line** (same for all three executables): `--func <id|name|all> --dim D --pop N --iters T --runs R --seed S --threads K --shift --scale <decay|literal> --levy-scale s --history <file> --no-header --list --selftest --help`.

**CUDA-only switch** (environment variable, not a flag): `HEO_CUDA_EVAL=thread|terms|auto` selects one thread per hare (default) or several threads per hare for the fitness evaluation; both give identical results (checked in Section 7).

**CSV columns** written to stdout (diagnostics go to stderr): `version,func,dim,pop,iters,shift,run,seed,threads,best,time_ms,compute_ms`. `time_ms` is end-to-end wall-clock per run as seen by the host; `compute_ms` is the algorithm loop only. Run *k* always uses seed `S + 1000003·k` in every back-end.

**Reproducibility.** The RNG is a pure function of `(seed, iteration, hare, dimension, stream)`, so values do not depend on thread scheduling or launch geometry. Candidate order and tie-breaking use a strict total order, so selection is deterministic.


In [ ]:
%%writefile heo_common.h
// heo_common.h - shared by the seq / OpenMP / CUDA builds.
// RNG, Levy flight, HEO update rule, selection helper and benchmark functions live here so
// all three versions execute the same math (counter-based RNG => results are reproducible and
// comparable across back-ends).
//
// ===========================================================================================
// PAPER-TO-CODE MAP (Alsamee & Ramezani, Sci. Rep. 15:26405, 2025) AND DOCUMENTED AMBIGUITIES
// ===========================================================================================
// Eq.(1)  D = L (X + V) r               -> the term  L*(x+v)*r  in heo_step(), per component.
//                                          L: fresh Levy step per (iteration, hare, dimension);
//                                          r ~ U(0,1) fresh per (iteration, hare, dimension).
// Eq.(4)  X' = X + (Maxit/t) V + L (X+V) r      -> heo_step(); V is the velocity *before* damping.
// Eq.(2)  V_{t+1} = (Maxit/t) V_t       -> NOT used literally (see A1).
// Pseudocode: initialise, then repeat { candidate for every hare by Eq.(4); merge old + new;
//             sort; keep the first N; first = best }       -> elitist selection (see A6).
//
// A1 Velocity update. Eq.(2) taken literally multiplies |V| by Maxit/t >= 1 every step, so V would
//    grow without bound, which contradicts "distance reduction". The experiments section states
//    "velocity reduction coefficient and damping are 1 and 0.95". Choice: V' = damp * V with
//    damp = 0.95; the reduction coefficient (vrc = 1) multiplies the position coefficient below.
// A2 Position coefficient. Eq.(3)/(4) use Maxit/t. Taken literally it equals T at t=1 and 1 at t=T:
//    it never drops below 1 and is huge early on (positions then saturate at the bounds). Both
//    readings are implemented and selected at run time with  --scale literal|decay :
//      decay   (DEFAULT, kept from the earlier version of this notebook): coef = vrc*(T-t+1)/T,
//              decreasing 1 -> 1/T, matching the text's "distance between hare and wolf is
//              gradually reduced";
//      literal : coef = vrc*T/t  (the equation exactly as printed).
//    The default is a documented interpretation, NOT the paper's printed equation.
// A3 Levy flight. The paper gives no algorithm or parameters. Choice: Mantegna's algorithm,
//    beta = 1.5, step = u / |v|^(1/beta), u ~ N(0, sigma_u^2), v ~ N(0,1), unscaled
//    (--levy-scale multiplies it). |v| is floored at 1e-12 to keep the step finite.
// A4 Initialisation. The pseudocode says "initialise random position for wolf V" (one wolf), yet
//    Eq.(4) uses a velocity V_{j,i} per hare. Choice: every hare gets its own position AND velocity,
//    both uniform in [lo, hi]^D from independent RNG streams.
// A5 Boundaries. The paper does not mention bound handling. Choice: clamp the new POSITION to
//    [lo, hi] (saturation). Velocity is not clamped (it only shrinks, by damping).
// A6 Selection. Parents and candidates are merged, sorted ascending by fitness and the best N are
//    kept. Ties: a parent beats a candidate; among parents / among candidates the lower index wins.
//    Because parents are always kept when better, the best fitness is monotonically non-increasing.
// A7 Iteration index. t runs 1..T (t = 0 would divide by zero in Maxit/t); Maxit = T = iters.
// A8 Benchmark typos in the paper (standard forms are used): Schwefel F18 lists fmin as
//    -418.9829 x 5 for d = 30 (here the function is offset so fmin = 0); Easom lacks a square on the
//    second exponent term; Beale's third term prints x1*x3^3; Schaffer N.2 is printed with a
//    non-standard layout. CEC 2015 F1-F15 need official shift/rotation data and are not implemented.
// A9 Noisy objective. F17 (Quartic) adds U[0,1) noise once per evaluation. A candidate's noisy value
//    is stored and elitism keeps it, so the reported best is the stored (noisy) fitness.
//
// FLOATING-POINT NOTE. Seq and OpenMP execute identical arithmetic (build with -ffp-contract=off) and
// agree bit for bit. CUDA uses its own sin/cos/log/exp/pow, which can differ from the host libm by
// ulps, so CUDA can differ at rounding level (and, rarely, through a changed selection order).
// CUB's radix sort orders -0.0 before +0.0 while '<' treats them as equal. Easom hits exactly +-0 on its
// plateau, so the CUDA kernels store fitness + 0.0 (maps -0.0 to +0.0) to keep the selection order identical.
#pragma once
#include <cstdint>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <string>
#include <vector>

#ifdef __CUDACC__
#define HD __host__ __device__
#else
#define HD
#endif

#ifdef USE_FLOAT
typedef float real;          // storage type of positions/velocities (fitness stays double)
#else
typedef double real;
#endif

#define HEO_PI 3.14159265358979323846

// ---------------------------------------------------------------- RNG
// Stateless counter-based RNG (splitmix64 finaliser). u01(seed, a, b, c) -> uniform in the OPEN
// interval (0,1): never exactly 0 or 1, so log() below is always finite. Same value on CPU and GPU.
HD inline uint64_t mix64(uint64_t z) {
  z += 0x9E3779B97F4A7C15ULL;
  z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ULL;
  z = (z ^ (z >> 27)) * 0x94D049BB133111EBULL;
  return z ^ (z >> 31);
}
HD inline double u01(uint64_t seed, uint64_t a, uint64_t b, uint64_t c) {
  uint64_t h = mix64(seed ^ mix64(a ^ mix64(b ^ mix64(c))));
  return ((double)(h >> 11) + 0.5) * (1.0 / 9007199254740992.0);     // (k + 0.5) / 2^53
}
// streams: 0=r  1..4=Levy  5=init x  6=init v  7=shift  9=quartic noise

// Levy flight step, Mantegna's algorithm, beta = 1.5 (A3)
HD inline double levy_step(uint64_t seed, uint64_t t, uint64_t e, double sigma) {
  double u1 = u01(seed, t, e, 1), u2 = u01(seed, t, e, 2);
  double u3 = u01(seed, t, e, 3), u4 = u01(seed, t, e, 4);
  double nu = sqrt(-2.0 * log(u1)) * cos(2.0 * HEO_PI * u2) * sigma;   // u ~ N(0, sigma^2)  (Box-Muller)
  double nv = sqrt(-2.0 * log(u3)) * cos(2.0 * HEO_PI * u4);           // v ~ N(0, 1)
  double an = fabs(nv); if (an < 1e-12) an = 1e-12;
  return nu / pow(an, 1.0 / 1.5);
}
static inline double levy_sigma_host(double beta = 1.5) {   // sigma_u(beta) ~ 0.69657 for beta = 1.5
  return pow(tgamma(1 + beta) * sin(HEO_PI * beta / 2) /
             (tgamma((1 + beta) / 2) * beta * pow(2.0, (beta - 1) / 2)), 1 / beta);
}

// ---------------------------------------------------------------- HEO update (Eq. 4)
// scale_mode 0 = "decay"  : coef = vrc*(T-t+1)/T   (default; see A2)
// scale_mode 1 = "literal": coef = vrc*Maxit/t     (paper equation as printed)
// X' = clamp( X + coef*V + L*(X+V)*r , lo, hi )   and   V' = damp*V          (A1, A5)
HD inline void heo_step(real x, real v, int t, int T, double vrc, double damp, int scale_mode,
                        double lo, double hi, real L, real r, real* xn, real* vn) {
  real coef = (scale_mode == 1) ? (real)(vrc * (double)T / (double)t)
                                : (real)(vrc * (double)(T - t + 1) / (double)T);
  real xx = x + coef * v + L * (x + v) * r;
  if (xx < (real)lo) xx = (real)lo;
  if (xx > (real)hi) xx = (real)hi;
  *xn = xx;
  *vn = (real)(damp * (double)v);
}

// ---------------------------------------------------------------- elitist selection helper (A6)
// A = parents in ascending order, B = candidates in ascending order, both of length n; A wins ties.
// merge_corank(k): how many of the first k merged elements come from A (binary search, O(log n)).
// merge_select(k): the k-th (0-based) merged element, k < n:  *fromA tells which list, *pos its index.
// Every output rank is independent, so OpenMP threads / CUDA threads can each take one rank.
template <class FA, class FB>
HD inline int merge_corank(int k, int n, FA a, FB b) {
  int lo = k > n ? k - n : 0, hi = k < n ? k : n;
  while (lo < hi) {
    int mid = lo + ((hi - lo) >> 1);
    if (a(mid) <= b(k - 1 - mid)) lo = mid + 1; else hi = mid;
  }
  return lo;
}
template <class FA, class FB>
HD inline void merge_select(int k, int n, FA a, FB b, bool* fromA, int* pos) {
  int i = merge_corank(k, n, a, b), j = k - i;           // for k < n both i < n and j < n
  if (a(i) <= b(j)) { *fromA = true; *pos = i; } else { *fromA = false; *pos = j; }
}

// ---------------------------------------------------------------- benchmarks
struct FuncInfo { const char* name; double lo, hi, fmin; int fixed_dim; int shiftable; };
static const FuncInfo FUNCS[] = {
  {"F16_StepSphere",  -100,  100, 0,          0, 1},
  {"F17_Quartic",     -1.28, 1.28, 0,         0, 1},   // + U[0,1) noise (A9)
  {"F18_Schwefel",    -500,  500, 0,          0, 0},   // offset so fmin = 0 (A8)
  {"F19_Rastrigin",   -5.12, 5.12, 0,         0, 1},
  {"Ackley",          -32,   32,  0,          0, 1},
  {"Griewank",        -600,  600, 0,          0, 1},
  {"Rosenbrock",      -30,   30,  0,          0, 0},
  {"F24_Bohachevsky", -100,  100, 0,          2, 0},
  {"F25_Booth",       -10,   10,  0,          2, 0},
  {"F26_Matyas",      -10,   10,  0,          2, 0},
  {"F27_Easom",       -100,  100, -1,         2, 0},
  {"F28_Beale",       -4.5,  4.5, 0,          2, 0},
  {"F29_SchafferN2",  -100,  100, 0,          2, 0},
  {"F30_SixHumpCamel",-2,    2,   -1.0316285, 2, 0},
};
static const int NFUNCS = (int)(sizeof(FUNCS) / sizeof(FUNCS[0]));

HD inline uint64_t shift_seed(int fid) { return 0xC0FFEEULL + (uint64_t)fid; }

// Wraps a coordinate accessor and (optionally) applies a fixed shift o_i
// (|o_i| <= 40% of half-range) so the optimum is NOT at the origin.
template <class A> struct ShiftAcc {
  A a; int on; uint64_t sseed; double half;
  HD double operator()(int i) const {
    double z = (double)a(i);
    if (on) z -= 0.4 * (2.0 * u01(sseed, 0, (uint64_t)i, 7) - 1.0) * half;
    return z;
  }
};

template <class A>
HD inline double bench_eval(int id, int D, A z, double noise) {
  double s = 0.0;
  switch (id) {
    case 0: for (int i = 0; i < D; i++) { double f = floor(z(i) + 0.5); s += f * f; } return s;
    case 1: for (int i = 0; i < D; i++) { double v = z(i); s += (i + 1) * v * v * v * v; } return s + noise;
    case 2: s = 418.9828872724337 * D;
            for (int i = 0; i < D; i++) { double v = z(i); s -= v * sin(sqrt(fabs(v))); } return s;
    case 3: for (int i = 0; i < D; i++) { double v = z(i); s += v * v - 10.0 * cos(2 * HEO_PI * v) + 10.0; } return s;
    case 4: { double s1 = 0, s2 = 0;
            for (int i = 0; i < D; i++) { double v = z(i); s1 += v * v; s2 += cos(2 * HEO_PI * v); }
            return -20.0 * exp(-0.2 * sqrt(s1 / D)) - exp(s2 / D) + 20.0 + 2.718281828459045; }
    case 5: { double p = 1.0;
            for (int i = 0; i < D; i++) { double v = z(i); s += v * v; p *= cos(v / sqrt(i + 1.0)); }
            return 1.0 + s / 4000.0 - p; }
    case 6: for (int i = 0; i < D - 1; i++) { double a = z(i), b = z(i + 1);
              s += 100.0 * (b - a * a) * (b - a * a) + (a - 1.0) * (a - 1.0); } return s;
    case 7: { double x = z(0), y = z(1);
            return x * x + 2 * y * y - 0.3 * cos(3 * HEO_PI * x) - 0.4 * cos(4 * HEO_PI * y) + 0.7; }
    case 8: { double x = z(0), y = z(1); double a = x + 2 * y - 7, b = 2 * x + y - 5; return a * a + b * b; }
    case 9: { double x = z(0), y = z(1); return 0.26 * (x * x + y * y) - 0.48 * x * y; }
    case 10:{ double x = z(0), y = z(1);
            return -cos(x) * cos(y) * exp(-((x - HEO_PI) * (x - HEO_PI) + (y - HEO_PI) * (y - HEO_PI))); }
    case 11:{ double x = z(0), y = z(1);
            double a = 1.5 - x + x * y, b = 2.25 - x + x * y * y, c = 2.625 - x + x * y * y * y;
            return a * a + b * b + c * c; }
    case 12:{ double x = z(0), y = z(1); double r2 = x * x + y * y; double sn = sin(sqrt(r2));
            double d = 1.0 + 0.001 * r2; return 0.5 + (sn * sn - 0.5) / (d * d); }
    case 13:{ double x = z(0), y = z(1);
            return 4 * x * x - 2.1 * x * x * x * x + x * x * x * x * x * x / 3.0 + x * y + (-4 + 4 * y * y) * y * y; }
  }
  return 0.0;
}

// ---------------------------------------------------------------- params / run info
struct Params {
  int fid = 0, dim = 30, pop = 30, iters = 500, runs = 10, shift = 0;
  uint64_t seed = 42;
  double vrc = 1.0, damp = 0.95;       // paper: velocity reduction coeff = 1, damping = 0.95
  double levy_scale = 1.0;             // paper uses raw L (scale 1); tunable via --levy-scale
  int scale_mode = 0;                  // 0 = decay (default), 1 = literal Maxit/t   (A2)
  int block = 256;                     // CUDA threads per block (ignored by CPU back-ends)
  double lo = 0, hi = 0, sigma = 0;
};

// Optional extra output of heo_run: compute-only time and best-so-far history.
struct RunInfo {
  double compute_ms = -1.0;                 // algorithm loop only (CUDA events on GPU, clock on CPU)
  std::vector<double>* history = nullptr;   // if set: history[0] = best after init, history[t] = after iteration t
};

// Returns an empty string if p is usable, otherwise a message describing the problem.
static inline std::string heo_validate(const Params& p) {
  if (p.fid < 0 || p.fid >= NFUNCS) return "function id out of range";
  if (p.dim < 1) return "dim must be >= 1";
  if (p.pop < 1) return "pop must be >= 1";
  if (p.iters < 1) return "iters must be >= 1";
  if (p.runs < 1) return "runs must be >= 1";
  if (p.block < 32 || p.block > 1024 || (p.block % 32)) return "block must be a multiple of 32 in [32,1024]";
  if (!(p.damp >= 0.0) || !(p.vrc >= 0.0) || !(p.levy_scale >= 0.0)) return "damp, vrc and levy-scale must be >= 0";
  if ((double)p.pop * 2.0 * (double)p.dim > 4.0e9) return "pop*dim too large (2*pop*dim must be <= 4e9 elements)";
  return "";
}


In [ ]:
%%writefile heo_main.h
// heo_main.h - common CLI + timing driver, included at the bottom of every back-end. A back-end defines
//   #define VERSION_NAME "seq"|"omp"|"cuda"
//   static int    heo_backend_init(int threads);   // returns the thread count to report (0 = not applicable)
//   static double heo_run(const Params&, uint64_t seed, RunInfo* info);   // returns best fitness
// and then includes this file. stdout carries ONLY the CSV (or --list / --selftest text); every
// diagnostic goes to stderr, so a notebook can parse stdout directly.
#pragma once
#include "heo_common.h"
#include <chrono>
#include <algorithm>
#include <cctype>
#include <cerrno>
#include <random>

struct SelfAcc { const double* p; HD double operator()(int i) const { return p[i]; } };

static void usage() {
  fprintf(stderr,
    "usage: heo_%s [options]\n"
    "  --func <id|name|all>   benchmark function (default all; see --list)\n"
    "  --dim D --pop N --iters T --runs R --seed S     (defaults 30 / 30 / 500 / 10 / 42)\n"
    "  --threads K            OpenMP threads (ignored by seq and cuda; default: OMP_NUM_THREADS / all cores)\n"
    "  --shift                move the optimum away from the origin (functions that support it)\n"
    "  --scale decay|literal  position coefficient: (T-t+1)/T (default) or Maxit/t as printed in the paper\n"
    "  --levy-scale s         multiply the Levy step (default 1)\n"
    "  --block B              CUDA threads per block, multiple of 32 (default 256; ignored on CPU)\n"
    "  --history <file>       also write best-so-far per iteration for run 0 (untimed extra run)\n"
    "  --no-header  --list  --selftest  --help\n"
    "run k uses seed S + 1000003*k in every back-end; CSV: version,func,dim,pop,iters,shift,run,seed,threads,best,time_ms,compute_ms\n",
    VERSION_NAME);
}
[[noreturn]] static void die(const std::string& msg) {
  fprintf(stderr, "error: %s\n", msg.c_str()); usage(); exit(2);
}
static long long parse_ll(const std::string& opt, const char* s, long long lo, long long hi) {
  char* end = nullptr; errno = 0; long long v = strtoll(s, &end, 10);
  if (end == s || *end != '\0' || errno) die(opt + ": '" + s + "' is not an integer");
  if (v < lo || v > hi) die(opt + ": " + s + " is out of range [" + std::to_string(lo) + ", " + std::to_string(hi) + "]");
  return v;
}
static double parse_dbl(const std::string& opt, const char* s) {
  char* end = nullptr; errno = 0; double v = strtod(s, &end);
  if (end == s || *end != '\0' || errno || !std::isfinite(v)) die(opt + ": '" + s + "' is not a finite number");
  return v;
}
static std::string lower(std::string s) { for (auto& c : s) c = (char)std::tolower((unsigned char)c); return s; }
// accepts a numeric id, the full name (any case) or the short tag before '_' (e.g. F19)
static int find_func(const std::string& s) {
  char* end = nullptr; long v = strtol(s.c_str(), &end, 10);
  if (end != s.c_str() && *end == '\0') return (v >= 0 && v < NFUNCS) ? (int)v : -1;
  std::string l = lower(s);
  for (int k = 0; k < NFUNCS; k++) {
    std::string n = lower(FUNCS[k].name), tag = n.substr(0, n.find('_'));
    if (l == n || l == tag) return k;
  }
  return -1;
}

// ------------------------------------------------------------------ --selftest (host-side checks)
static int heo_selftest() {
  int fails = 0;
  auto chk = [&](bool ok, const char* what, double val) {
    printf("%s  %-52s %.6g\n", ok ? "PASS" : "FAIL", what, val); if (!ok) fails++; };
  { const int n = 1000000; double s1 = 0, s2 = 0, mn = 1, mx = 0;
    for (int i = 0; i < n; i++) { double u = u01(7, (uint64_t)(i / 1000), (uint64_t)(i % 1000), 0);
      s1 += u; s2 += u * u; mn = std::min(mn, u); mx = std::max(mx, u); }
    double m = s1 / n, v = s2 / n - m * m;
    chk(std::fabs(m - 0.5) < 2e-3, "u01 mean ~ 0.5", m);
    chk(std::fabs(v - 1.0 / 12) < 2e-3, "u01 variance ~ 1/12", v);
    chk(mn > 0.0 && mx < 1.0, "u01 stays inside the open interval (0,1)", mn); }
  { double sg = levy_sigma_host(); chk(std::fabs(sg - 0.6966) < 1e-3, "Levy sigma_u(beta=1.5) ~ 0.6966", sg);
    int bad = 0; for (int i = 0; i < 200000; i++) { double l = levy_step(11, 1 + i % 50, (uint64_t)i, sg); if (!std::isfinite(l)) bad++; }
    chk(bad == 0, "Levy steps are finite (count of non-finite)", bad); }
  { double z[8]; auto ev = [&](int id, int D) { return bench_eval(id, D, SelfAcc{z}, 0.0); };
    for (int i = 0; i < 8; i++) { z[i] = 0; }
    chk(ev(0, 8) == 0 && ev(3, 8) == 0 && std::fabs(ev(4, 8)) < 1e-12 && ev(5, 8) == 0, "f(0)=0 for StepSphere/Rastrigin/Ackley/Griewank", ev(4, 8));
    chk(ev(7, 2) == 0 && ev(9, 2) == 0 && ev(12, 2) == 0 && ev(1, 8) == 0, "f(0)=0 for Bohachevsky/Matyas/Schaffer2/Quartic(no noise)", 0);
    for (int i = 0; i < 8; i++) { z[i] = 420.9687462275036; }
    chk(std::fabs(ev(2, 8)) < 1e-9, "Schwefel at 420.9687 ~ 0", ev(2, 8));
    for (int i = 0; i < 8; i++) { z[i] = 1; }
    chk(ev(6, 8) == 0, "Rosenbrock at (1,..,1) = 0", ev(6, 8));
    z[0] = 1; z[1] = 3;                                   chk(ev(8, 2) == 0, "Booth at (1,3) = 0", ev(8, 2));
    z[0] = HEO_PI; z[1] = HEO_PI;                         chk(std::fabs(ev(10, 2) + 1) < 1e-12, "Easom at (pi,pi) = -1", ev(10, 2));
    z[0] = 3; z[1] = 0.5;                                 chk(ev(11, 2) == 0, "Beale at (3,0.5) = 0", ev(11, 2));
    z[0] = 0.08984201; z[1] = -0.71265640;                chk(std::fabs(ev(13, 2) + 1.0316285) < 1e-6, "Six-hump camel at its minimiser", ev(13, 2)); }
  { real xn, vn; heo_step(90, 50, 1, 10, 1.0, 0.95, 0, -100, 100, 0.7f, 0.3f, &xn, &vn);
    chk(xn == (real)100 && std::fabs((double)vn - 47.5) < 1e-9, "heo_step clamps position, damps velocity", (double)vn);
    heo_step(1.5, -2.0, 3, 10, 1.0, 0.95, 1, -100, 100, 0.7, 0.3, &xn, &vn);
    chk(std::fabs((double)xn - (1.5 + (10.0 / 3.0) * -2.0 + 0.7 * (1.5 - 2.0) * 0.3)) < 1e-12, "literal Maxit/t equals paper Eq.(4)", (double)xn); }
  { std::mt19937 g(1); long bad = 0, trials = 0;
    for (int tr = 0; tr < 2000; tr++) {
      int n = 1 + (int)(g() % 40); std::vector<double> A(n), B(n);
      for (auto& x : A) x = (double)(g() % 8);
      for (auto& x : B) x = (double)(g() % 8);
      std::sort(A.begin(), A.end()); std::sort(B.begin(), B.end());
      std::vector<std::pair<bool, int>> ref; int i = 0, j = 0;
      while ((int)ref.size() < n) { if (i < n && (j >= n || A[i] <= B[j])) ref.push_back({true, i++}); else ref.push_back({false, j++}); }
      for (int k = 0; k < n; k++) { bool fa; int pos;
        merge_select(k, n, [&](int q) { return A[q]; }, [&](int q) { return B[q]; }, &fa, &pos);
        trials++; if (fa != ref[k].first || pos != ref[k].second) bad++; } }
    chk(bad == 0, "merge_select == reference merge (mismatches)", (double)bad); }
  printf("%s (%d failure%s)\n", fails ? "SELFTEST FAILED" : "SELFTEST PASSED", fails, fails == 1 ? "" : "s");
  return fails ? 1 : 0;
}

int main(int argc, char** argv) {
  Params p; std::string func = "all", hist_file; bool header = true, dim_given = false; int threads_req = 0;
  for (int a = 1; a < argc; a++) {
    std::string s = argv[a];
    auto need = [&]() -> const char* { if (a + 1 >= argc) die("missing value for " + s); return argv[++a]; };
    if (s == "--func") func = need();
    else if (s == "--dim") { p.dim = (int)parse_ll(s, need(), 1, 100000000); dim_given = true; }
    else if (s == "--pop") p.pop = (int)parse_ll(s, need(), 1, 100000000);
    else if (s == "--iters") p.iters = (int)parse_ll(s, need(), 1, 1000000000);
    else if (s == "--runs") p.runs = (int)parse_ll(s, need(), 1, 1000000);
    else if (s == "--seed") p.seed = (uint64_t)parse_ll(s, need(), 0, 9223372036854775807LL);
    else if (s == "--threads") threads_req = (int)parse_ll(s, need(), 1, 4096);
    else if (s == "--levy-scale") p.levy_scale = parse_dbl(s, need());
    else if (s == "--block") p.block = (int)parse_ll(s, need(), 32, 1024);
    else if (s == "--scale") { std::string v = lower(need());
      if (v == "decay") p.scale_mode = 0; else if (v == "literal") p.scale_mode = 1; else die("--scale expects decay or literal"); }
    else if (s == "--history") hist_file = need();
    else if (s == "--shift") p.shift = 1;
    else if (s == "--no-header") header = false;
    else if (s == "--help" || s == "-h") { usage(); return 0; }
    else if (s == "--selftest") return heo_selftest();
    else if (s == "--list") {
      printf("id,name,lo,hi,fmin,fixed_dim,shiftable\n");
      for (int k = 0; k < NFUNCS; k++) printf("%d,%s,%g,%g,%.10g,%d,%d\n", k, FUNCS[k].name, FUNCS[k].lo, FUNCS[k].hi, FUNCS[k].fmin, FUNCS[k].fixed_dim, FUNCS[k].shiftable);
      return 0; }
    else die("unknown option '" + s + "'");
  }
  { std::string err = heo_validate(p); if (!err.empty()) die(err); }
  int f0 = 0, f1 = NFUNCS;
  if (func != "all") { f0 = find_func(func); if (f0 < 0) die("unknown function '" + func + "' (use --list)"); f1 = f0 + 1; }
  FILE* hf = nullptr;
  if (!hist_file.empty()) {
    hf = fopen(hist_file.c_str(), "w"); if (!hf) die("cannot open history file '" + hist_file + "'");
    fprintf(hf, "version,func,dim,pop,iters,shift,seed,iter,best\n");
  }
  int threads = heo_backend_init(threads_req);
  if (header) printf("version,func,dim,pop,iters,shift,run,seed,threads,best,time_ms,compute_ms\n");
  for (int fid = f0; fid < f1; fid++) {
    Params q = p; q.fid = fid; q.lo = FUNCS[fid].lo; q.hi = FUNCS[fid].hi;
    if (FUNCS[fid].fixed_dim) {
      if (dim_given && p.dim != FUNCS[fid].fixed_dim)
        fprintf(stderr, "note: %s is %d-dimensional by definition; --dim %d ignored for it\n", FUNCS[fid].name, FUNCS[fid].fixed_dim, p.dim);
      q.dim = FUNCS[fid].fixed_dim;
    }
    if (p.shift && !FUNCS[fid].shiftable) fprintf(stderr, "note: %s does not support --shift; running unshifted\n", FUNCS[fid].name);
    q.shift = p.shift && FUNCS[fid].shiftable;
    q.sigma = levy_sigma_host();
    Params w = q; w.iters = 2; heo_run(w, 1, nullptr);                // untimed warm-up at the SAME size (workspace, JIT, threads, CUDA context)
    if (hf) {                                                         // untimed convergence-history run: same seed as run 0
      std::vector<double> h; RunInfo hi; hi.history = &h; heo_run(q, p.seed, &hi);
      for (size_t t = 0; t < h.size(); t++)
        fprintf(hf, "%s,%s,%d,%d,%d,%d,%llu,%zu,%.17g\n", VERSION_NAME, FUNCS[fid].name, q.dim, q.pop, q.iters, q.shift, (unsigned long long)p.seed, t, h[t]);
    }
    for (int r = 0; r < p.runs; r++) {
      uint64_t seed = p.seed + 1000003ULL * (uint64_t)r;
      RunInfo info;
      auto t0 = std::chrono::steady_clock::now();
      double f = heo_run(q, seed, &info);
      auto t1 = std::chrono::steady_clock::now();
      double ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
      if (!std::isfinite(f)) fprintf(stderr, "warning: non-finite best fitness (func %s, run %d)\n", FUNCS[fid].name, r);
      printf("%s,%s,%d,%d,%d,%d,%d,%llu,%d,%.17g,%.4f,", VERSION_NAME, FUNCS[fid].name, q.dim, q.pop, q.iters, q.shift, r,
             (unsigned long long)seed, threads, f, ms);
      if (info.compute_ms >= 0) printf("%.4f\n", info.compute_ms); else printf("nan\n");
      fflush(stdout);
    }
  }
  if (hf) fclose(hf);
  return 0;
}


In [ ]:
%%writefile heo_seq.cpp
// Sequential reference implementation of Hare Escape Optimization (single thread).
// All algorithm details and documented paper ambiguities are in heo_common.h.
//
// Per iteration:  (1) candidate + fitness for every hare   (2) sort the N candidates by (fitness, index)
//                 (3) merge with the already-sorted parents, keep the best N (parents win ties)
//                 (4) gather the survivors.
// Steps 2-3 give exactly the same survivors, in the same order, as sorting all 2N elements with the tie
// rule "lower index first" (parents precede candidates), but cost O(N log N) instead of O(2N log 2N).
// Row j of the parent block is the hare that owns RNG elements j*D+i; after the first iteration the
// parents are stored in sorted order, so row index == rank (hence `porder` is the identity from then on).
#define VERSION_NAME "seq"
#include "heo_common.h"
#include <algorithm>
#include <numeric>
#include <chrono>

struct RowAcc { const real* p; HD real operator()(int i) const { return p[i]; } };
static int heo_backend_init(int) { return 1; }

// Workspace is allocated once and reused by every run (the untimed warm-up in heo_main.h sizes it),
// so the timed runs contain no heap allocation.
struct Workspace {
  std::vector<real> pos, vel, pos2, vel2;      // rows [0,N) parents, rows [N,2N) candidates; row = hare, D reals
  std::vector<double> fit, fit2;               // fit[0,N) parents, fit[N,2N) candidates ; fit2 = survivors
  std::vector<int> porder, cidx;               // parents in ascending order / candidates in ascending order
  void reserve(size_t N, size_t D) {
    size_t cnt = 2 * N * D;
    if (pos.size() < cnt) { pos.resize(cnt); vel.resize(cnt); pos2.resize(cnt); vel2.resize(cnt); }
    if (fit.size() < 2 * N) { fit.resize(2 * N); fit2.resize(2 * N); }
    if (porder.size() < N) { porder.resize(N); cidx.resize(N); }
  }
};

static double heo_run(const Params& p, uint64_t seed, RunInfo* info) {
  const int N = p.pop, D = p.dim, T = p.iters;
  static Workspace ws; ws.reserve((size_t)N, (size_t)D);
  real *pos = ws.pos.data(), *vel = ws.vel.data(), *pos2 = ws.pos2.data(), *vel2 = ws.vel2.data();
  double *fit = ws.fit.data(), *fit2 = ws.fit2.data();
  int *porder = ws.porder.data(), *cidx = ws.cidx.data();
  const uint64_t sseed = shift_seed(p.fid); const double half = 0.5 * (p.hi - p.lo);
  auto eval = [&](const real* row, double noise) {
    ShiftAcc<RowAcc> s{RowAcc{row}, p.shift, sseed, half};
    return bench_eval(p.fid, D, s, noise); };
  std::vector<double>* hist = info ? info->history : nullptr;
  if (hist) { hist->clear(); hist->reserve((size_t)T + 1); }

  auto c0 = std::chrono::steady_clock::now();
  for (int j = 0; j < N; j++) {                                   // initial population (A4)
    for (int i = 0; i < D; i++) {
      uint64_t e = (uint64_t)j * D + i;
      pos[(size_t)j * D + i] = (real)(p.lo + (p.hi - p.lo) * u01(seed, 0, e, 5));
      vel[(size_t)j * D + i] = (real)(p.lo + (p.hi - p.lo) * u01(seed, 0, e, 6));
    }
    fit[j] = eval(&pos[(size_t)j * D], u01(seed, 0, j, 9));
  }
  std::iota(porder, porder + N, 0);                               // parents ordered by (fitness, index); rows stay put
  std::sort(porder, porder + N, [&](int a, int b) { return fit[a] < fit[b] || (fit[a] == fit[b] && a < b); });
  if (hist) hist->push_back(fit[porder[0]]);

  for (int t = 1; t <= T; t++) {
    for (int j = 0; j < N; j++) {                                 // candidate for each hare (Eq. 4)
      for (int i = 0; i < D; i++) {
        uint64_t e = (uint64_t)j * D + i;
        real L = (real)(levy_step(seed, t, e, p.sigma) * p.levy_scale);
        real r = (real)u01(seed, t, e, 0);
        heo_step(pos[(size_t)j * D + i], vel[(size_t)j * D + i], t, T, p.vrc, p.damp, p.scale_mode, p.lo, p.hi, L, r,
                 &pos[(size_t)(N + j) * D + i], &vel[(size_t)(N + j) * D + i]);
      }
      fit[N + j] = eval(&pos[(size_t)(N + j) * D], u01(seed, t, j, 9));
    }
    std::iota(cidx, cidx + N, 0);                                 // sort candidates by (fitness, index)
    std::sort(cidx, cidx + N, [&](int a, int b) { return fit[N + a] < fit[N + b] || (fit[N + a] == fit[N + b] && a < b); });
    int a = 0, b = 0;                                             // merge: keep best N, parents win ties (A6)
    for (int k = 0; k < N; k++) {
      bool fromA = (b >= N) || (a < N && fit[porder[a]] <= fit[N + cidx[b]]);
      int src = fromA ? porder[a++] : N + cidx[b++];
      memcpy(&pos2[(size_t)k * D], &pos[(size_t)src * D], D * sizeof(real));
      memcpy(&vel2[(size_t)k * D], &vel[(size_t)src * D], D * sizeof(real));
      fit2[k] = fit[src];
    }
    std::swap(pos, pos2); std::swap(vel, vel2); std::swap(fit, fit2);
    std::iota(porder, porder + N, 0);                             // survivors are stored in rank order
    if (hist) hist->push_back(fit[0]);
  }
  double best = fit[0];                                           // elitism: fit[0] is the best-so-far (never worse than any earlier best)
  auto c1 = std::chrono::steady_clock::now();
  if (info) info->compute_ms = std::chrono::duration<double, std::milli>(c1 - c0).count();
  // The swaps above may leave ws.pos/ws.pos2 exchanged, which is fine: both are scratch buffers of equal size.
  return best;
}
#include "heo_main.h"


In [ ]:
%%writefile heo_omp.cpp
// OpenMP implementation of HEO. Same algorithm and arithmetic as heo_seq.cpp (see heo_common.h), so results are
// bit-identical to the sequential reference for ANY thread count:
//  * every random number is a pure function of (seed, iteration, hare, dimension, stream) => independent of
//    which thread computes it and of the schedule;
//  * hares are independent within an iteration (each writes only its own candidate row) => no races, no atomics,
//    no critical sections;
//  * sorting uses a strict total order (fitness, index) => the same permutation whatever the thread count;
//  * each output rank k of the merge/selection is computed independently with a binary search (merge_select),
//    fused with the row gather => one parallel region instead of two.
// Per iteration: parallel update+evaluation | parallel sort of the N candidates | parallel selection+gather.
// schedule(static): every hare costs the same (D transcendental evaluations), so static chunks have the least
// scheduling overhead and keep each thread on contiguous rows (cache locality, little false sharing).
#define VERSION_NAME "omp"
#include "heo_common.h"
#include <algorithm>
#include <numeric>
#include <chrono>
#include <parallel/algorithm>   // __gnu_parallel::sort (libstdc++, needs -fopenmp)
#include <omp.h>

struct RowAcc { const real* p; HD real operator()(int i) const { return p[i]; } };

static int heo_backend_init(int threads) {
  if (threads > 0) omp_set_num_threads(threads);       // --threads wins over OMP_NUM_THREADS
  int n = omp_get_max_threads();
  fprintf(stderr, "[omp] threads = %d\n", n);
  return n;
}

struct Workspace {                                       // allocated once, reused by every run (see heo_seq.cpp)
  std::vector<real> pos, vel, pos2, vel2;
  std::vector<double> fit, fit2;
  std::vector<int> porder, cidx;
  void reserve(size_t N, size_t D) {
    size_t cnt = 2 * N * D;
    if (pos.size() < cnt) { pos.resize(cnt); vel.resize(cnt); pos2.resize(cnt); vel2.resize(cnt); }
    if (fit.size() < 2 * N) { fit.resize(2 * N); fit2.resize(2 * N); }
    if (porder.size() < N) { porder.resize(N); cidx.resize(N); }
  }
};

static double heo_run(const Params& p, uint64_t seed, RunInfo* info) {
  const int N = p.pop, D = p.dim, T = p.iters;
  static Workspace ws; ws.reserve((size_t)N, (size_t)D);
  real *pos = ws.pos.data(), *vel = ws.vel.data(), *pos2 = ws.pos2.data(), *vel2 = ws.vel2.data();
  double *fit = ws.fit.data(), *fit2 = ws.fit2.data();
  int *porder = ws.porder.data(), *cidx = ws.cidx.data();
  const uint64_t sseed = shift_seed(p.fid); const double half = 0.5 * (p.hi - p.lo);
  std::vector<double>* hist = info ? info->history : nullptr;
  if (hist) { hist->clear(); hist->reserve((size_t)T + 1); }

  auto c0 = std::chrono::steady_clock::now();
  #pragma omp parallel for schedule(static)
  for (int j = 0; j < N; j++) {                                     // initial population (A4)
    for (int i = 0; i < D; i++) {
      uint64_t e = (uint64_t)j * D + i;
      pos[(size_t)j * D + i] = (real)(p.lo + (p.hi - p.lo) * u01(seed, 0, e, 5));
      vel[(size_t)j * D + i] = (real)(p.lo + (p.hi - p.lo) * u01(seed, 0, e, 6));
    }
    ShiftAcc<RowAcc> s{RowAcc{&pos[(size_t)j * D]}, p.shift, sseed, half};
    fit[j] = bench_eval(p.fid, D, s, u01(seed, 0, j, 9));
  }
  std::iota(porder, porder + N, 0);
  __gnu_parallel::sort(porder, porder + N, [&](int a, int b) { return fit[a] < fit[b] || (fit[a] == fit[b] && a < b); });
  if (hist) hist->push_back(fit[porder[0]]);

  for (int t = 1; t <= T; t++) {
    #pragma omp parallel for schedule(static)
    for (int j = 0; j < N; j++) {                                   // candidate + fitness, independent per hare
      for (int i = 0; i < D; i++) {
        uint64_t e = (uint64_t)j * D + i;
        real L = (real)(levy_step(seed, t, e, p.sigma) * p.levy_scale);
        real r = (real)u01(seed, t, e, 0);
        heo_step(pos[(size_t)j * D + i], vel[(size_t)j * D + i], t, T, p.vrc, p.damp, p.scale_mode, p.lo, p.hi, L, r,
                 &pos[(size_t)(N + j) * D + i], &vel[(size_t)(N + j) * D + i]);
      }
      ShiftAcc<RowAcc> s{RowAcc{&pos[(size_t)(N + j) * D]}, p.shift, sseed, half};
      fit[N + j] = bench_eval(p.fid, D, s, u01(seed, t, j, 9));
    }
    std::iota(cidx, cidx + N, 0);                                   // parallel sort of the N candidates (strict total order)
    __gnu_parallel::sort(cidx, cidx + N, [&](int a, int b) { return fit[N + a] < fit[N + b] || (fit[N + a] == fit[N + b] && a < b); });
    #pragma omp parallel for schedule(static)
    for (int k = 0; k < N; k++) {                                   // survivor k: binary-search merge, then copy its row
      bool fromA; int idx;
      merge_select(k, N, [&](int q) { return fit[porder[q]]; }, [&](int q) { return fit[N + cidx[q]]; }, &fromA, &idx);
      int src = fromA ? porder[idx] : N + cidx[idx];
      memcpy(&pos2[(size_t)k * D], &pos[(size_t)src * D], D * sizeof(real));
      memcpy(&vel2[(size_t)k * D], &vel[(size_t)src * D], D * sizeof(real));
      fit2[k] = fit[src];
    }
    std::swap(pos, pos2); std::swap(vel, vel2); std::swap(fit, fit2);
    if (t == 1) std::iota(porder, porder + N, 0);                   // survivors are in rank order from now on
    if (hist) hist->push_back(fit[0]);
  }
  double best = fit[0];
  auto c1 = std::chrono::steady_clock::now();
  if (info) info->compute_ms = std::chrono::duration<double, std::milli>(c1 - c0).count();
  return best;
}
#include "heo_main.h"


In [ ]:
%%writefile heo_cuda.cu
// CUDA implementation of HEO. Same algorithm and RNG streams as heo_seq.cpp (details: heo_common.h).
//
// Data layout: dimension-major, element (hare j, dim i) at [i*M + j], M = 2N. Columns [0,N) = parents,
// [N,2N) = candidates. Consecutive threads handle consecutive hares j, so every position/velocity access in the
// update, evaluation and gather kernels is coalesced.
//
// Everything stays on the device for the whole run. The ONLY host<->device transfers are the final best fitness
// (8 bytes) and, if --history is used, the history array. Buffers are allocated once and reused by every run
// (the untimed warm-up in heo_main.h sizes them).
//
// Per iteration (all on the default stream):
//   k_update   one thread per (hare, dimension): candidate (Eq. 4), coalesced
//   evaluation one thread per hare (k_eval)  -- or --  one thread per (hare, dimension) (k_terms) followed by
//              k_reduce (see "ONE THREAD PER HARE vs SEVERAL" below)
//   CUB radix sort of the N candidate fitnesses (stable => ties keep ascending index)
//   k_select   one thread per survivor rank: binary-search merge of sorted parents and sorted candidates
//   k_gather   one thread per (hare, dimension): copy the survivors (coalesced writes)
// Compared with sorting all 2N elements, only N are sorted and the former k_iota launch is gone (the iota array is
// constant); the sort of the parents happens once, before iteration 1.
//
// ONE THREAD PER HARE vs SEVERAL (evaluation). Update and gather already use one thread per (hare, dimension).
// Evaluation sums D terms per hare. With one thread per hare (default, HEO_CUDA_EVAL=thread) only N threads run,
// which under-fills a GPU when N is small (a T4 holds ~40k resident threads). A block-wide reduction would use
// more threads but changes the floating-point summation order, breaking bit-level parity with the CPU versions.
// HEO_CUDA_EVAL=terms keeps the order: all N*D terms (the expensive sin/cos/exp calls) are computed in parallel by
// k_terms, then k_reduce adds them left to right exactly as bench_eval does (one thread per hare, coalesced
// reads). It applies to the scalable functions (ids 0-6); others always use k_eval. HEO_CUDA_EVAL=auto picks terms
// when N < 8192 and D >= 64. Which mode is faster depends on the GPU and the sizes: measure it (Section 7).
//
// Timing: compute_ms = CUDA events around the whole algorithm loop (first kernel -> last kernel). The caller's
// time_ms additionally includes host launch latency, the final synchronisation and the 8-byte copy back.
#define VERSION_NAME "cuda"
#include "heo_common.h"
#include <cuda_runtime.h>
#include <cub/cub.cuh>
#include <utility>

#define CK(x) do { cudaError_t e_ = (x); if (e_ != cudaSuccess) { \
  fprintf(stderr, "CUDA error '%s' at %s:%d\n", cudaGetErrorString(e_), __FILE__, __LINE__); exit(1); } } while (0)

struct ColAcc { const real* p; int stride; HD real operator()(int i) const { return p[(size_t)i * stride]; } };
struct ParentKey { const double* fit; const int* order; HD double operator()(int q) const { return fit[order[q]]; } };
struct SortedKey { const double* key; HD double operator()(int q) const { return key[q]; } };

__global__ void k_init(real* pos, real* vel, int N, int D, int M, uint64_t seed, double lo, double hi) {
  size_t tid = blockIdx.x * (size_t)blockDim.x + threadIdx.x; if (tid >= (size_t)N * D) return;
  int j = (int)(tid % N), i = (int)(tid / N); uint64_t e = (uint64_t)j * D + i;
  pos[(size_t)i * M + j] = (real)(lo + (hi - lo) * u01(seed, 0, e, 5));
  vel[(size_t)i * M + j] = (real)(lo + (hi - lo) * u01(seed, 0, e, 6));
}
__global__ void k_update(real* pos, real* vel, int N, int D, int M, uint64_t seed, int t, int T,
                         double vrc, double damp, int scale_mode, double lo, double hi, double sigma, double lscale) {
  size_t tid = blockIdx.x * (size_t)blockDim.x + threadIdx.x; if (tid >= (size_t)N * D) return;
  int j = (int)(tid % N), i = (int)(tid / N); uint64_t e = (uint64_t)j * D + i;
  size_t a = (size_t)i * M + j, b = (size_t)i * M + N + j;
  real L = (real)(levy_step(seed, t, e, sigma) * lscale);
  real r = (real)u01(seed, t, e, 0);
  heo_step(pos[a], vel[a], t, T, vrc, damp, scale_mode, lo, hi, L, r, &pos[b], &vel[b]);
}
// evaluation, one thread per hare: columns [off, off+N)
__global__ void k_eval(const real* pos, double* fit, int off, int N, int D, int M, int fid, int shift,
                       double lo, double hi, uint64_t seed, int t) {
  int j = blockIdx.x * blockDim.x + threadIdx.x; if (j >= N) return;
  ColAcc a{pos + off + j, M};
  ShiftAcc<ColAcc> s{a, shift, shift_seed(fid), 0.5 * (hi - lo)};
  fit[off + j] = bench_eval(fid, D, s, u01(seed, t, j, 9)) + 0.0;   // +0.0 turns -0.0 into +0.0 (see k_reduce)
}
// evaluation, several threads per hare: term kernel + order-preserving reduction (ids 0-6 only).
// Expressions are copied from bench_eval so each term, and the left-to-right sum, is bit-identical.
__global__ void k_terms(const real* pos, double* t1, double* t2, int off, int N, int D, int M, int fid, int shift,
                        double lo, double hi) {
  size_t tid = blockIdx.x * (size_t)blockDim.x + threadIdx.x; if (tid >= (size_t)N * D) return;
  int j = (int)(tid % N), i = (int)(tid / N);
  ColAcc a{pos + off + j, M};
  ShiftAcc<ColAcc> z{a, shift, shift_seed(fid), 0.5 * (hi - lo)};
  double v = z(i), x1 = 0.0, x2 = 0.0;
  switch (fid) {
    case 0: { double f = floor(v + 0.5); x1 = f * f; break; }
    case 1: x1 = (i + 1) * v * v * v * v; break;
    case 2: x1 = v * sin(sqrt(fabs(v))); break;
    case 3: x1 = v * v - 10.0 * cos(2 * HEO_PI * v) + 10.0; break;
    case 4: x1 = v * v; x2 = cos(2 * HEO_PI * v); break;
    case 5: x1 = v * v; x2 = cos(v / sqrt(i + 1.0)); break;
    case 6: if (i < D - 1) { double aa = v, bb = z(i + 1);
              x1 = 100.0 * (bb - aa * aa) * (bb - aa * aa) + (aa - 1.0) * (aa - 1.0); } break;
  }
  t1[(size_t)i * N + j] = x1; t2[(size_t)i * N + j] = x2;
}
__global__ void k_reduce(const double* t1, const double* t2, double* fit, int off, int N, int D, int fid,
                         uint64_t seed, int t) {
  int j = blockIdx.x * blockDim.x + threadIdx.x; if (j >= N) return;
  double s = 0.0, s2 = 0.0, p = 1.0;
  if (fid == 2) s = 418.9828872724337 * D;
  int n = (fid == 6) ? D - 1 : D;
  for (int i = 0; i < n; i++) {
    double a = t1[(size_t)i * N + j];
    if (fid == 2) s -= a; else s += a;
    if (fid == 4) s2 += t2[(size_t)i * N + j];
    if (fid == 5) p *= t2[(size_t)i * N + j];
  }
  double f;
  switch (fid) {
    case 1:  f = s + u01(seed, t, j, 9); break;
    case 4:  f = -20.0 * exp(-0.2 * sqrt(s / D)) - exp(s2 / D) + 20.0 + 2.718281828459045; break;
    case 5:  f = 1.0 + s / 4000.0 - p; break;
    default: f = s; break;
  }
  // CUB's radix sort orders -0.0 before +0.0, while the CPU versions treat them as equal (tie -> lower index).
  // Easom underflows to exactly +-0 on its plateau, so normalise: x + 0.0 maps -0.0 to +0.0 and changes no other value.
  fit[off + j] = f + 0.0;
}
__global__ void k_iota(int* idx, int n) { int k = blockIdx.x * blockDim.x + threadIdx.x; if (k < n) idx[k] = k; }
// survivor k = k-th element of merge(parents, candidates), parents win ties (A6)
__global__ void k_select(const double* fit, double* fit2, const int* porder, const double* ckey, const int* cidx,
                         int* sel, int N) {
  int k = blockIdx.x * blockDim.x + threadIdx.x; if (k >= N) return;
  bool fromA; int idx;
  merge_select(k, N, ParentKey{fit, porder}, SortedKey{ckey}, &fromA, &idx);
  int src = fromA ? porder[idx] : N + cidx[idx];
  sel[k] = src; fit2[k] = fit[src];
}
__global__ void k_gather(const real* pos, const real* vel, real* pos2, real* vel2, const int* sel, int N, int D, int M) {
  size_t tid = blockIdx.x * (size_t)blockDim.x + threadIdx.x; if (tid >= (size_t)N * D) return;
  int j = (int)(tid % N), i = (int)(tid / N); int s = sel[j];
  pos2[(size_t)i * M + j] = pos[(size_t)i * M + s];
  vel2[(size_t)i * M + j] = vel[(size_t)i * M + s];
}
__global__ void k_record(const double* fit, const int* order, double* hist, int t) {   // hist[t] = best so far
  hist[t] = fit[order ? order[0] : 0];
}

// ---------------------------------------------------------------- persistent device workspace
struct Workspace {
  size_t capN = 0, capD = 0, capTerm = 0, capHist = 0, tmpb = 0;
  real *pos = 0, *vel = 0, *pos2 = 0, *vel2 = 0;
  double *fit = 0, *fit2 = 0, *ckey = 0, *pkey = 0, *t1 = 0, *t2 = 0, *hist = 0;
  int *iota_ = 0, *porder = 0, *cidx = 0, *sel = 0;
  void* tmp = 0; cudaEvent_t e0, e1; bool ev = false;
  void free_main() {
    cudaFree(pos); cudaFree(vel); cudaFree(pos2); cudaFree(vel2); cudaFree(fit); cudaFree(fit2); cudaFree(ckey);
    cudaFree(pkey); cudaFree(iota_); cudaFree(porder); cudaFree(cidx); cudaFree(sel);
    pos = vel = pos2 = vel2 = 0; fit = fit2 = ckey = pkey = 0; iota_ = porder = cidx = sel = 0;
  }
  void reserve(int N, int D, bool terms, int Hn) {
    if (!ev) { CK(cudaEventCreate(&e0)); CK(cudaEventCreate(&e1)); ev = true; }
    if ((size_t)N > capN || (size_t)D > capD) {
      free_main(); capN = N > (int)capN ? N : capN; capD = D > (int)capD ? D : capD;
      size_t cnt = 2 * capN * capD;
      CK(cudaMalloc(&pos, cnt * sizeof(real)));  CK(cudaMalloc(&vel, cnt * sizeof(real)));
      CK(cudaMalloc(&pos2, cnt * sizeof(real))); CK(cudaMalloc(&vel2, cnt * sizeof(real)));
      CK(cudaMalloc(&fit, 2 * capN * sizeof(double))); CK(cudaMalloc(&fit2, 2 * capN * sizeof(double)));
      CK(cudaMalloc(&ckey, capN * sizeof(double)));    CK(cudaMalloc(&pkey, capN * sizeof(double)));
      CK(cudaMalloc(&iota_, capN * sizeof(int)));      CK(cudaMalloc(&porder, capN * sizeof(int)));
      CK(cudaMalloc(&cidx, capN * sizeof(int)));       CK(cudaMalloc(&sel, capN * sizeof(int)));
    }
    size_t need = 0;                                            // CUB temp storage for sorting N (key,int) pairs
    CK(cub::DeviceRadixSort::SortPairs(nullptr, need, fit, ckey, iota_, cidx, N));
    if (need > tmpb) { cudaFree(tmp); tmp = 0; CK(cudaMalloc(&tmp, need)); tmpb = need; }
    if (terms && (size_t)N * D > capTerm) {
      cudaFree(t1); cudaFree(t2); capTerm = (size_t)N * D;
      CK(cudaMalloc(&t1, capTerm * sizeof(double))); CK(cudaMalloc(&t2, capTerm * sizeof(double)));
    }
    if (Hn > 0 && (size_t)Hn > capHist) { cudaFree(hist); capHist = Hn; CK(cudaMalloc(&hist, capHist * sizeof(double))); }
  }
};
static Workspace ws;

static const char* g_eval_env = "thread";
static int heo_backend_init(int) {
  CK(cudaFree(0));
  cudaDeviceProp pr; CK(cudaGetDeviceProperties(&pr, 0));
  const char* e = getenv("HEO_CUDA_EVAL"); g_eval_env = (e && *e) ? e : "thread";
  if (strcmp(g_eval_env, "thread") && strcmp(g_eval_env, "terms") && strcmp(g_eval_env, "auto")) {
    fprintf(stderr, "error: HEO_CUDA_EVAL must be thread, terms or auto (got '%s')\n", g_eval_env); exit(2); }
  fprintf(stderr, "[cuda] device = %s (%d SMs) | eval mode = %s\n", pr.name, pr.multiProcessorCount, g_eval_env);
  return 0;
}

static double heo_run(const Params& p, uint64_t seed, RunInfo* info) {
  const int N = p.pop, D = p.dim, M = 2 * N, T = p.iters, B = p.block;
  std::vector<double>* hist = info ? info->history : nullptr;
  bool terms = false;
  if (p.fid <= 6 && strcmp(g_eval_env, "thread") != 0)
    terms = strcmp(g_eval_env, "terms") == 0 || (N < 8192 && D >= 64);
  ws.reserve(N, D, terms, hist ? T + 1 : 0);
  real *pos = ws.pos, *vel = ws.vel, *pos2 = ws.pos2, *vel2 = ws.vel2; double *fit = ws.fit, *fit2 = ws.fit2;
  const unsigned gND = (unsigned)(((size_t)N * D + B - 1) / B), gN = (unsigned)((N + B - 1) / B);

  auto evaluate = [&](int off, int t) {                              // fitness of columns [off, off+N)
    if (terms) {
      k_terms<<<gND, B>>>(pos, ws.t1, ws.t2, off, N, D, M, p.fid, p.shift, p.lo, p.hi);
      k_reduce<<<gN, B>>>(ws.t1, ws.t2, fit, off, N, D, p.fid, seed, t);
    } else {
      k_eval<<<gN, B>>>(pos, fit, off, N, D, M, p.fid, p.shift, p.lo, p.hi, seed, t);
    }
  };

  CK(cudaEventRecord(ws.e0));
  k_iota<<<gN, B>>>(ws.iota_, N);
  k_init<<<gND, B>>>(pos, vel, N, D, M, seed, p.lo, p.hi);
  evaluate(0, 0);
  CK(cub::DeviceRadixSort::SortPairs(ws.tmp, ws.tmpb, fit, ws.pkey, ws.iota_, ws.porder, N));   // parents by (fitness, index)
  if (hist) k_record<<<1, 1>>>(fit, ws.porder, ws.hist, 0);
  for (int t = 1; t <= T; t++) {
    k_update<<<gND, B>>>(pos, vel, N, D, M, seed, t, T, p.vrc, p.damp, p.scale_mode, p.lo, p.hi, p.sigma, p.levy_scale);
    evaluate(N, t);
    CK(cub::DeviceRadixSort::SortPairs(ws.tmp, ws.tmpb, fit + N, ws.ckey, ws.iota_, ws.cidx, N));  // stable: ties keep index order
    const int* po = (t == 1) ? ws.porder : ws.iota_;                  // survivors are in rank order after iteration 1
    k_select<<<gN, B>>>(fit, fit2, po, ws.ckey, ws.cidx, ws.sel, N);
    k_gather<<<gND, B>>>(pos, vel, pos2, vel2, ws.sel, N, D, M);
    std::swap(pos, pos2); std::swap(vel, vel2); std::swap(fit, fit2);
    if (hist) k_record<<<1, 1>>>(fit, nullptr, ws.hist, t);
    CK(cudaGetLastError());
  }
  CK(cudaEventRecord(ws.e1));
  CK(cudaEventSynchronize(ws.e1));
  if (info) { float ms = 0.f; CK(cudaEventElapsedTime(&ms, ws.e0, ws.e1)); info->compute_ms = (double)ms; }
  double best; CK(cudaMemcpy(&best, fit, sizeof(double), cudaMemcpyDeviceToHost));   // survivors sorted: fit[0] is the best
  if (hist) { hist->resize((size_t)T + 1); CK(cudaMemcpy(hist->data(), ws.hist, ((size_t)T + 1) * sizeof(double), cudaMemcpyDeviceToHost)); }
  return best;
}
#include "heo_main.h"


## 3. Compile sequential, OpenMP and CUDA programs

| Target | Command | Why these flags |
|---|---|---|
| Sequential | `g++ -O3 -std=c++17 -ffp-contract=off heo_seq.cpp -o heo_seq` | `-O3` optimisation; `-std=c++17` for the language features used; `-ffp-contract=off` forbids fused multiply-add so every `a*b+c` rounds twice, as on every other back-end |
| OpenMP | `g++ -O3 -std=c++17 -ffp-contract=off -fopenmp heo_omp.cpp -o heo_omp` | as above, plus `-fopenmp` (enables the `#pragma omp` directives and libstdc++'s parallel `sort`) |
| CUDA | `nvcc -O3 -std=c++17 -arch=sm_XX --fmad=false heo_cuda.cu -o heo_cuda` | `-arch` is the GPU's compute capability from Section 1 (code for the wrong architecture will not launch); `--fmad=false` is nvcc's equivalent of `-ffp-contract=off` |

Deliberately **not** used: `-ffast-math` / `--use_fast_math` (they reassociate sums and replace `sin`, `exp`, ... by approximations, which would invalidate the parity check) and `-march=native` (it enables FMA instructions and ties the binary to one CPU).

After compiling, the next cell (a) stops if any build fails, (b) checks that each executable exists, (c) runs every executable's `--selftest`, and (d) runs a small identical command on all back-ends to confirm they accept the same arguments, write the same CSV columns and agree numerically. The benchmark cells only run if all of this succeeds.


In [ ]:
# Cell 11: compile all back-ends, then smoke-test them. Raises (stopping the notebook) on any failure.
def sh(cmd):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip() or r.stderr.strip():
        print((r.stdout + r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError(f"Compilation failed (exit code {r.returncode}): {' '.join(cmd)}\n"
                           "Fix the error shown above (or re-run the source cells) before continuing.")

SOURCES = ["heo_common.h", "heo_main.h", "heo_seq.cpp", "heo_omp.cpp"] + (["heo_cuda.cu"] if HAS_GPU else [])
missing = [f for f in SOURCES if not os.path.exists(f)]
if missing:
    raise RuntimeError(f"Missing source files {missing}: run the Section 2 cells first.")

CPU_FLAGS = ["-O3", "-std=c++17", "-ffp-contract=off"]
sh(["g++"] + CPU_FLAGS + ["heo_seq.cpp", "-o", "heo_seq"])
sh(["g++"] + CPU_FLAGS + ["-fopenmp", "heo_omp.cpp", "-o", "heo_omp"])
if HAS_GPU:
    sh(["nvcc", "-O3", "-std=c++17"] + ([f"-arch={CUDA_ARCH}"] if CUDA_ARCH else []) + ["--fmad=false", "heo_cuda.cu", "-o", "heo_cuda"])
else:
    print("(CUDA build skipped: no usable GPU/nvcc in this runtime)")

# ---- (b) executables exist
IMPLS = ["seq", "omp"] + (["cuda"] if HAS_GPU else [])
for v in IMPLS:
    if not (os.path.isfile(f"./heo_{v}") and os.access(f"./heo_{v}", os.X_OK)):
        raise RuntimeError(f"./heo_{v} was not produced.")

# ---- (c) host-side self-tests (RNG, Levy step, benchmark optima, update rule, merge selection)
for v in IMPLS:
    r = subprocess.run([f"./heo_{v}", "--selftest"], capture_output=True, text=True)
    last = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else "(no output)"
    print(f"selftest heo_{v}: {last}")
    if r.returncode != 0:
        print(r.stdout)
        raise RuntimeError(f"heo_{v} --selftest failed.")

# ---- (d) identical arguments on every back-end: same columns, same seeds, numerically consistent results
SMOKE = ["--func", "all", "--dim", "20", "--pop", "64", "--iters", "30", "--runs", "2", "--seed", "7"]
smoke = {}
for v in IMPLS:
    r = subprocess.run([f"./heo_{v}"] + SMOKE + ["--threads", "2"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"heo_{v} rejected the shared arguments {' '.join(SMOKE)}:\n{r.stderr}")
    smoke[v] = pd.read_csv(io.StringIO(r.stdout), float_precision="round_trip")   # exact parse: pandas' default parser loses ~1e-13
cols = list(smoke["seq"].columns)
for v in IMPLS:
    if list(smoke[v].columns) != cols:
        raise RuntimeError(f"heo_{v} writes columns {list(smoke[v].columns)}, expected {cols}")
    if not (smoke[v][["func", "run", "seed"]].equals(smoke["seq"][["func", "run", "seed"]])):
        raise RuntimeError(f"heo_{v} did not use the same function/run/seed sequence as the sequential reference.")
d_omp = (smoke["omp"].best - smoke["seq"].best).abs().max()
print(f"smoke parity OpenMP vs sequential: max |delta best| = {d_omp:.3e} (expected exactly 0)")
if d_omp != 0:
    raise RuntimeError("OpenMP does not reproduce the sequential reference bit for bit; do not benchmark until this is fixed.")
if HAS_GPU:
    rel = ((smoke["cuda"].best - smoke["seq"].best).abs() / (1.0 + smoke["seq"].best.abs()))
    print(f"smoke parity CUDA vs sequential: max relative delta = {rel.max():.3e}; rows above 1e-6: {(rel > 1e-6).sum()} / {len(rel)}")
    if (rel > 1e-6).mean() > 0.25:
        raise RuntimeError("CUDA disagrees with the sequential reference on many runs: a speedup would be meaningless.")
print("Build and smoke tests passed for:", ", ".join(IMPLS))


## 4. Benchmark methodology

**What is compared.** The three executables built above, on the same functions, dimensions, population sizes and iteration counts. Every configuration is executed by every back-end through the same command line.

**Repeated independent runs and matched seeds.** Each configuration is run `RUNS` times (≥ 20 in the `final` profile). Run *k* uses seed `S + 1000003·k`, and that seed is the same in every back-end. Because the RNG is counter-based, run *k* of the OpenMP and CUDA versions draws exactly the same random numbers as run *k* of the sequential reference, so differences in the final fitness are due to arithmetic, not luck, and paired statistical tests are valid.

**Configurations.** Several population sizes, dimensions and iteration counts are used (`CONFIGS` in the next cell). The 2-D functions F24–F30 are fixed at dimension 2 by definition; they are tiny and launch-overhead dominated, so they are reported separately from the scalable functions. A second pass uses `--shift` (optimum moved away from the origin) on one configuration to expose the origin bias of the update rule.

**Timing.** Two numbers are recorded per run and never mixed:
* `time_ms` – end-to-end wall-clock around one complete run as seen by the host (for CUDA this includes kernel launch latency, the final synchronisation and the copy of the best value back to the host);
* `compute_ms` – only the algorithm loop: CUDA events on the GPU, a clock around the loop on the CPU.

Both exclude an untimed warm-up run of identical size, which allocates the (reused) workspaces and starts the OpenMP threads and the CUDA context. One-time allocation is therefore *not* part of the timings; this is steady-state performance.

**Convergence.** For a few functions one extra, untimed run per back-end (same seed as run 0) records the best fitness after every iteration.

**What is not claimed.** Results are specific to the hardware printed in Section 1. Nothing in this notebook is pre-computed; if a stage cannot run it is skipped with a message.


In [ ]:
# Cell 13: benchmark configuration and execution.
import datetime
PROFILE = os.environ.get("HEO_PROFILE", "final")      # "smoke" (seconds), "standard" (minutes), "final" (>= 20 runs, tens of minutes)
PROFILES = {
    "smoke":    dict(runs=2,  shift_runs=2,  configs=[dict(dim=10, pop=32, iters=20), dict(dim=20, pop=64, iters=30)]),
    "standard": dict(runs=5,  shift_runs=5,  configs=[dict(dim=30, pop=256, iters=200), dict(dim=30, pop=1024, iters=500),
                                                      dict(dim=100, pop=256, iters=500), dict(dim=100, pop=1024, iters=200)]),
    "final":    dict(runs=20, shift_runs=20, configs=[dict(dim=30, pop=256, iters=200), dict(dim=30, pop=1024, iters=500),
                                                      dict(dim=100, pop=256, iters=500), dict(dim=100, pop=1024, iters=200)]),
}
if PROFILE not in PROFILES:
    raise ValueError(f"PROFILE must be one of {list(PROFILES)}, got {PROFILE!r}")
RUNS, SHIFT_RUNS, CONFIGS = PROFILES[PROFILE]["runs"], PROFILES[PROFILE]["shift_runs"], PROFILES[PROFILE]["configs"]
BASE_SEED = 42                                         # run k uses BASE_SEED + 1000003*k in every back-end
OMP_THREADS = os.cpu_count()                           # change to test another thread count
SHIFT_CONFIG = CONFIGS[min(1, len(CONFIGS) - 1)]       # configuration used for the shifted-optimum study
CONV_FUNCS = ["F19_Rastrigin", "Ackley", "Griewank"]  # functions whose convergence history is recorded
SCALABLE = ["F16_StepSphere", "F17_Quartic", "F18_Schwefel", "F19_Rastrigin", "Ackley", "Griewank", "Rosenbrock"]
SHIFTABLE_IDS = [0, 1, 3, 4, 5]
LABEL = {"seq": "CPU (sequential)", "omp": "OpenMP", "cuda": "CUDA GPU"}
if RUNS < 20:
    print(f"NOTE: profile '{PROFILE}' uses {RUNS} runs per configuration; use PROFILE='final' (20 runs) for reported results.")

FUNC_TABLE = pd.read_csv(io.StringIO(subprocess.run(["./heo_seq", "--list"], capture_output=True, text=True).stdout)).set_index("name")

def run(exe, func="all", extra=(), dim=None, pop=None, iters=None, runs=None, threads=None, env_extra=None):
    cmd = [f"./heo_{exe}", "--func", str(func), "--dim", str(dim), "--pop", str(pop), "--iters", str(iters),
           "--runs", str(runs), "--seed", str(BASE_SEED), "--threads", str(threads or OMP_THREADS)] + list(extra)
    env = os.environ.copy(); env.update(env_extra or {})
    p = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if p.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed (exit {p.returncode}):\n{p.stderr}")
    return pd.read_csv(io.StringIO(p.stdout), float_precision="round_trip")    # exact parse (pandas' default loses ~1e-13)

def cfg_label(c): return f"D{c['dim']}_N{c['pop']}_T{c['iters']}"

frames, skipped, t_start = [], [], time.time()
for c in CONFIGS:
    for v in IMPLS:
        if v == "cuda" and not fits_on_gpu(c["pop"], c["dim"]):
            skipped.append((cfg_label(c), v, "does not fit in free GPU memory")); continue
        df = run(v, **c, runs=RUNS); df["cfg"] = cfg_label(c); frames.append(df)
        print(f"[{time.time() - t_start:7.1f}s] {cfg_label(c)} {v}: {len(df)} runs")
for v in IMPLS:                                        # shifted optimum, one configuration
    if v == "cuda" and not fits_on_gpu(SHIFT_CONFIG["pop"], SHIFT_CONFIG["dim"]): continue
    for fid in SHIFTABLE_IDS:
        df = run(v, func=fid, extra=["--shift"], **SHIFT_CONFIG, runs=SHIFT_RUNS); df["cfg"] = cfg_label(SHIFT_CONFIG); frames.append(df)
    print(f"[{time.time() - t_start:7.1f}s] shifted study {v} done")

results = pd.concat(frames, ignore_index=True)
results["seconds"] = results["time_ms"] / 1000.0            # end-to-end
results["compute_seconds"] = results["compute_ms"] / 1000.0  # algorithm loop only (CUDA events on GPU)
results["implementation"] = results["version"].map(LABEL)
results.to_csv("heo_raw_trials.csv", index=False)

# ---- convergence histories (untimed extra run, same seed as run 0)
hist_frames = []
cc = CONFIGS[min(1, len(CONFIGS) - 1)]
for v in IMPLS:
    if v == "cuda" and not fits_on_gpu(cc["pop"], cc["dim"]): continue
    for fn in CONV_FUNCS:
        hf = f"heo_hist_{v}_{fn}.csv"
        run(v, func=fn, **cc, runs=1, extra=["--history", hf])
        hist_frames.append(pd.read_csv(hf, float_precision="round_trip"))
history = pd.concat(hist_frames, ignore_index=True)

RUN_META = dict(profile=PROFILE, runs=RUNS, shift_runs=SHIFT_RUNS, configs=CONFIGS, shift_config=SHIFT_CONFIG,
                base_seed=BASE_SEED, omp_threads=OMP_THREADS, impls=IMPLS, skipped=skipped,
                finished=datetime.datetime.now().isoformat(timespec="seconds"), elapsed_s=round(time.time() - t_start, 1))
display(results.head(8))
print("rows:", len(results), "| configurations:", [cfg_label(c) for c in CONFIGS], "| OpenMP threads:", OMP_THREADS,
      "| total benchmark time: %.1f s" % RUN_META["elapsed_s"])
if skipped: print("SKIPPED:", skipped)


## 5. Statistics, speedup, efficiency, overhead and parity

**Speedup** is `T_sequential / T_parallel`, computed from mean times over the repeated runs (> 1 means the parallel version is faster). It is reported twice: *end-to-end* (what a user waits for) and *compute-only* (the algorithm loop). The two differ for the GPU because end-to-end time also contains kernel-launch latency, the final synchronisation and the transfer of the result.

**Parallel efficiency** is `speedup / p`, where `p` is the number of OpenMP threads. Ideal scaling gives 1.0; values below 1 are caused by serial parts (the sort), memory bandwidth, thread start-up and load imbalance. It is **not defined for CUDA**, whose processing elements are not comparable to CPU threads, so the table shows `n/a` for the GPU.

**Runtime overhead** is the share of end-to-end time spent *outside* the algorithm loop, `(time − compute) / time`. For CPU versions it is almost zero; for CUDA it is the host-side launch/synchronisation/transfer cost, which dominates for small problems and explains why the GPU only wins once there is enough parallel work.

**Numerical parity check.** With matched seeds the OpenMP result must equal the sequential one **bit for bit** (same arithmetic, deterministic RNG, strict-total-order sorting). CUDA may differ at rounding level because device `sin/cos/exp/log/pow` are not bit-identical to the host library; the check reports the largest absolute and relative differences and how many runs exceed 1e-6. A large mismatch would make every speedup meaningless. CSV values are parsed with `float_precision="round_trip"` because pandas' default parser alone can introduce errors of about 1e-13.

**Statistical tests** (paired by seed, hence by run): Wilcoxon signed-rank for sequential vs each parallel version on final fitness and on runtime, and the Friedman test over all implementations (needs three, i.e. a GPU). When two result vectors are identical every difference is zero and the Wilcoxon test is undefined; the notebook then reports "identical" instead of inventing a p-value. **Success rate** is the fraction of runs ending within `SUCCESS_TOL` of the known optimum; Quartic is excluded because its objective contains random noise.


In [ ]:
# Cell 15: statistical analysis. Everything is computed from `results`; nothing is typed in.
from scipy import stats
SUCCESS_TOL = 1e-6                      # |best - fmin| <= SUCCESS_TOL counts as success (absolute)
FMIN = FUNC_TABLE["fmin"].to_dict()
GROUP = ["cfg", "shift", "func", "version"]

def q(x, p): return np.percentile(x, p)
def fit_stats(g):
    b, fn = g["best"].values, g.name[GROUP.index("func")]      # g.name = (cfg, shift, func, version)
    ok = np.nan if fn == "F17_Quartic" else float(np.mean(np.abs(b - FMIN[fn]) <= SUCCESS_TOL))
    return pd.Series(dict(n_runs=len(b), best=b.min(), mean=b.mean(), worst=b.max(), median=np.median(b),
                          std=b.std(ddof=1) if len(b) > 1 else np.nan, iqr=q(b, 75) - q(b, 25), success_rate=ok,
                          time_mean_s=g["seconds"].mean(), time_std_s=g["seconds"].std(ddof=1),
                          compute_mean_s=g["compute_seconds"].mean(), compute_std_s=g["compute_seconds"].std(ddof=1)))
summary = results.groupby(GROUP).apply(fit_stats, include_groups=False).reset_index()
summary["overhead_frac"] = (summary["time_mean_s"] - summary["compute_mean_s"]) / summary["time_mean_s"]

# ---- speedup, efficiency, overhead relative to the sequential reference
rows = []
for (cfg, sh, fn), g in summary.groupby(["cfg", "shift", "func"]):
    s = g[g.version == "seq"].iloc[0]
    for _, r in g[g.version != "seq"].iterrows():
        sp, spc = s.time_mean_s / r.time_mean_s, s.compute_mean_s / r.compute_mean_s
        rows.append(dict(cfg=cfg, shift=sh, func=fn, version=r.version, speedup_end_to_end=sp, speedup_compute=spc,
                         efficiency=(sp / OMP_THREADS) if r.version == "omp" else np.nan, overhead_frac=r.overhead_frac))
speedup = pd.DataFrame(rows)

print("=== Fitness statistics (unshifted; lower is better) ===")
show = summary[summary["shift"] == 0].copy()
for c in CONFIGS[-1:]:
    d = show[(show.cfg == cfg_label(c)) & show.func.isin(SCALABLE)]
    print(f"\nconfiguration {cfg_label(c)}, {RUNS} runs per cell"); display(d[["func", "version", "best", "mean", "worst", "median", "std", "iqr", "success_rate"]].round(6))
print("\n=== Runtime, speedup, efficiency, overhead (scalable functions, all configurations; efficiency = speedup / %d threads, n/a for CUDA) ===" % OMP_THREADS)
sp_view = (speedup[(speedup["shift"] == 0) & speedup.func.isin(SCALABLE)].groupby(["cfg", "version"])
           [["speedup_end_to_end", "speedup_compute", "efficiency", "overhead_frac"]].mean().round(3))
display(sp_view)

# ---- numerical parity against the sequential reference (matched seeds)
key = ["cfg", "shift", "func", "run"]
base = results[results.version == "seq"].set_index(key)["best"]
parity = []
for v in IMPLS[1:]:
    o = results[results.version == v].set_index(key)["best"]
    diff = (o - base).abs(); rel = diff / (1.0 + base.abs())
    parity.append(dict(version=v, runs=len(diff), bit_identical=int((o.values == base.loc[o.index].values).sum()),
                       max_abs=diff.max(), max_rel=rel.max(), runs_rel_gt_1e6=int((rel > 1e-6).sum())))
parity = pd.DataFrame(parity); print("\n=== Numerical parity vs sequential ==="); display(parity)
if len(parity) and (parity[parity.version == "omp"].max_abs.iloc[0] != 0):
    print("WARNING: OpenMP differs from the sequential reference. Investigate before trusting any speedup.")

# ---- Wilcoxon signed-rank (paired by run) and Friedman
def wilcoxon_rows(metric):
    out = []
    for (cfg, sh, fn), g in results.groupby(["cfg", "shift", "func"]):
        p = g.pivot(index="run", columns="version", values=metric)
        for v in [x for x in IMPLS if x != "seq"]:
            d = (p["seq"] - p[v]).values
            if np.all(d == 0): out.append(dict(cfg=cfg, shift=sh, func=fn, vs=v, n=len(d), p_value=np.nan, note="identical (all differences zero)"))
            elif len(d) < 6: out.append(dict(cfg=cfg, shift=sh, func=fn, vs=v, n=len(d), p_value=np.nan, note="too few runs"))
            else: out.append(dict(cfg=cfg, shift=sh, func=fn, vs=v, n=len(d), p_value=stats.wilcoxon(p["seq"], p[v]).pvalue, note=""))
    return pd.DataFrame(out)
w_fit, w_time = wilcoxon_rows("best"), wilcoxon_rows("seconds")
for name, w in (("final fitness", w_fit), ("end-to-end runtime", w_time)):
    tested = w.dropna(subset=["p_value"])
    print(f"\nWilcoxon signed-rank on {name}: {len(w)} paired comparisons, {len(w) - len(tested)} identical/untestable, "
          f"{int((tested.p_value < 0.05).sum())} of {len(tested)} testable with p < 0.05 (no multiplicity correction)")
    display(w.groupby(["vs", "note"], dropna=False).size().rename("comparisons").reset_index())

def friedman(metric):
    if len(IMPLS) < 3: return "needs >= 3 implementations (no GPU in this run)"
    p = results.pivot_table(index=["cfg", "shift", "func", "run"], columns="version", values=metric)[IMPLS].dropna()
    if (p.values == p.values[:, [0]]).all(): return "all implementations identical in every block"
    s, pv = stats.friedmanchisquare(*[p[v].values for v in IMPLS]); return f"chi2 = {s:.2f}, p = {pv:.3g} over {len(p)} blocks"
print("\nFriedman test, final fitness :", friedman("best"))
print("Friedman test, runtime       :", friedman("seconds"))

summary.to_csv("heo_summary.csv", index=False); speedup.to_csv("heo_speedup.csv", index=False)
parity.to_csv("heo_parity.csv", index=False); w_fit.to_csv("heo_wilcoxon_fitness.csv", index=False); w_time.to_csv("heo_wilcoxon_runtime.csv", index=False)
print("\nSaved heo_raw_trials.csv, heo_summary.csv, heo_speedup.csv, heo_parity.csv, heo_wilcoxon_*.csv")


## 6. Comparison graphs

All graphs are drawn from the tables computed above for the largest configuration (`PLOT_CFG`), unshifted optimum, scalable functions. Purpose of each figure:

1. **Runtime** – mean end-to-end and compute-only time per function (log scale, error bars = standard deviation over runs). Answers *how long does each back-end take?*
2. **Speedup** – sequential time divided by parallel time, end-to-end and compute-only. The dashed line at 1 marks break-even; the gap between the two panels is the host-side overhead of the GPU.
3. **Parallel efficiency** – OpenMP speedup divided by the thread count (1 = ideal). Not drawn for CUDA.
4. **Fitness quality** – distribution of the final fitness over the independent runs. Because of matched seeds, identical distributions confirm that parallelisation did not change the optimiser's behaviour. Panels are separate because objective scales differ by orders of magnitude.
5. **Convergence** – best-so-far fitness versus iteration for one seeded run; overlapping curves again confirm parity. Each panel is drawn only if its data exist.
6. **Optimum location** – the same optimiser with the optimum at the origin versus shifted away from it, which exposes the origin bias of the update term `L·(X+V)·r`.


In [ ]:
# Cell 17: performance graphs (all data come from `summary`, `speedup`, `results`, `history`).
from matplotlib.ticker import FuncFormatter
plain = FuncFormatter(lambda v, _: f"{v:g}")          # readable log-axis tick labels
PLOT_CFG = cfg_label(CONFIGS[-1])
COLORS = {"seq": "#4c72b0", "omp": "#dd8452", "cuda": "#55a868"}
S0 = summary[(summary["shift"] == 0) & (summary.cfg == PLOT_CFG)]
funcs = [f for f in SCALABLE if f in set(S0.func)]
x = np.arange(len(funcs)); w = 0.8 / len(IMPLS)
subtitle = f"{PLOT_CFG.replace('_', ', ')}, {RUNS} runs, OpenMP threads = {OMP_THREADS}"

def grouped(ax, col, err, ylabel, title):
    for k, v in enumerate(IMPLS):
        d = S0[S0.version == v].set_index("func").reindex(funcs)
        ax.bar(x + (k - (len(IMPLS) - 1) / 2) * w, d[col], w, yerr=d[err], capsize=2, label=LABEL[v], color=COLORS[v])
    ax.set_xticks(x); ax.set_xticklabels(funcs, rotation=30, ha="right"); ax.set_yscale("log")
    ax.yaxis.set_major_formatter(plain); ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis="y", alpha=0.3, which="both"); ax.legend()

# 1. runtime ---------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
grouped(ax[0], "time_mean_s", "time_std_s", "Mean time per run (s, log; lower is better)", "End-to-end time")
grouped(ax[1], "compute_mean_s", "compute_std_s", "", "Compute-only time (algorithm loop)")
fig.suptitle(f"Runtime comparison: Sequential vs OpenMP vs CUDA ({subtitle})"); plt.tight_layout(); plt.savefig("fig_runtime.png", dpi=150); plt.show()

# 2. speedup and 3. efficiency ----------------------------------------------
SP = speedup[(speedup["shift"] == 0) & (speedup.cfg == PLOT_CFG)]
par = [v for v in IMPLS if v != "seq"]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for a, col, ttl in ((ax[0], "speedup_end_to_end", "End-to-end speedup"), (ax[1], "speedup_compute", "Compute-only speedup")):
    for k, v in enumerate(par):
        d = SP[SP.version == v].set_index("func").reindex(funcs)
        a.bar(x + (k - (len(par) - 1) / 2) * 0.8 / len(par), d[col], 0.8 / len(par), label=LABEL[v], color=COLORS[v])
    a.axhline(1.0, ls="--", color="k", lw=1); a.set_xticks(x); a.set_xticklabels(funcs, rotation=30, ha="right")
    a.set_yscale("log"); a.yaxis.set_major_formatter(plain); a.set_title(ttl); a.grid(axis="y", alpha=0.3, which="both"); a.legend()
ax[0].set_ylabel("Speedup = T_sequential / T_parallel (log; > 1 faster)")
fig.suptitle(f"Speedup over the sequential reference ({subtitle})"); plt.tight_layout(); plt.savefig("fig_speedup.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(8, 4.2))
d = SP[SP.version == "omp"].set_index("func").reindex(funcs)
ax.bar(x, d["efficiency"], color=COLORS["omp"]); ax.axhline(1.0, ls="--", color="k", lw=1, label="ideal (1.0)")
ax.set_xticks(x); ax.set_xticklabels(funcs, rotation=30, ha="right"); ax.set_ylabel("Efficiency = speedup / threads")
ax.set_title(f"OpenMP parallel efficiency ({OMP_THREADS} threads, {PLOT_CFG.replace('_', ', ')})"); ax.grid(axis="y", alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig("fig_efficiency.png", dpi=150); plt.show()

# 4. fitness quality (boxplots over runs) -------------------------------------
R0 = results[(results["shift"] == 0) & (results.cfg == PLOT_CFG) & results.func.isin(funcs)]
ncols = 4; nrows = int(np.ceil(len(funcs) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.3 * nrows)); axes = np.atleast_1d(axes).ravel()
for a, fn in zip(axes, funcs):
    data = [R0[(R0.func == fn) & (R0.version == v)].best.values for v in IMPLS]
    bp = a.boxplot(data, patch_artist=True, tick_labels=[LABEL[v].split()[0] for v in IMPLS])
    for patch, v in zip(bp["boxes"], IMPLS): patch.set_facecolor(COLORS[v])
    a.set_title(fn, fontsize=10); a.set_ylabel("Final best fitness"); a.grid(axis="y", alpha=0.3)
for a in axes[len(funcs):]: a.set_visible(False)
fig.suptitle(f"Solution quality over {RUNS} matched-seed runs (lower is better; {subtitle})"); plt.tight_layout(); plt.savefig("fig_fitness.png", dpi=150); plt.show()

# 5. convergence ---------------------------------------------------------------
cf = [f for f in CONV_FUNCS if f in set(history.func)]
if cf:
    fig, axes = plt.subplots(1, len(cf), figsize=(5 * len(cf), 4.2)); axes = np.atleast_1d(axes)
    for a, fn in zip(axes, cf):
        for v, ls, lw in zip(IMPLS, ["-", "--", ":"], [3, 2, 1.5]):
            h = history[(history.func == fn) & (history.version == v)]
            if len(h): a.plot(h.iter, h.best, ls=ls, lw=lw, color=COLORS[v], label=LABEL[v])
        a.set_yscale("log"); a.set_xlabel("Iteration"); a.set_ylabel("Best fitness so far (log)"); a.set_title(fn); a.grid(alpha=0.3, which="both"); a.legend()
    hc = history.iloc[0]
    fig.suptitle(f"Convergence of HEO, one seeded run (D={hc.dim}, N={hc.pop}, T={hc.iters}); overlapping curves = matching behaviour")
    plt.tight_layout(); plt.savefig("fig_convergence.png", dpi=150); plt.show()
else:
    print("No convergence data available; convergence figure skipped.")

# 6. optimum at origin vs shifted ----------------------------------------------
sc = cfg_label(SHIFT_CONFIG)
sh_f = [f for f in SCALABLE if ((summary.cfg == sc) & (summary["shift"] == 1) & (summary.func == f) & (summary.version == "seq")).any()]
if sh_f:
    pick = lambda s, f: summary[(summary.cfg == sc) & (summary["shift"] == s) & (summary.func == f) & (summary.version == "seq")]["mean"].iloc[0]
    cmpdf = pd.DataFrame({"optimum at origin": [pick(0, f) for f in sh_f], "shifted optimum": [pick(1, f) for f in sh_f]}, index=sh_f)
    ax = cmpdf.plot(kind="bar", figsize=(9, 4), logy=True)
    ax.set_title(f"HEO sensitivity to optimum location ({sc.replace('_', ', ')}, sequential, mean of {SHIFT_RUNS} runs)")
    ax.set_ylabel("Mean best fitness (log; lower is better)"); ax.grid(axis="y", alpha=0.3, which="both"); plt.xticks(rotation=20)
    plt.tight_layout(); plt.savefig("fig_shift.png", dpi=150); plt.show()


## 7. Scaling experiments

A GPU only wins once there is enough parallel work to hide launch latency, and a CPU only gains from threads when each thread receives enough work. The sweeps below use Rastrigin (`SCALE_FUNC`) and measure where each back-end overtakes the sequential code.

* **Strong scaling** – the problem size is fixed and the number of OpenMP threads `p` grows. Ideal: time `∝ 1/p`, speedup `= p`, efficiency `= 1`. It is limited by the serial fraction (Amdahl's law): here the sort and the thread start-up. Measured by the *thread-scaling* experiment.
* **Weak scaling** – the work per thread is fixed (population `N ∝ p`) and `p` grows. Ideal: constant time, so weak-scaling efficiency `= T(1) / T(p) = 1`. It shows whether the code can solve a proportionally larger problem in the same time.
* **Population scaling** – `N` grows at fixed dimension and iterations. Work grows linearly with `N` (the paper reports linear complexity); parallel versions need large `N` to fill threads or GPU cores.
* **Dimension scaling** – `D` grows at fixed `N`. Work grows linearly with `D` per hare; the GPU gains twice because the update and gather kernels parallelise over `N·D` elements.
* **CUDA evaluation modes** (GPU runs only) – one thread per hare versus several threads per hare for the fitness evaluation (`HEO_CUDA_EVAL=thread|terms`), compared on compute time and checked for bit-identical results.

Only measured values are plotted. Points that do not fit in GPU memory are skipped and reported.


In [ ]:
# Cell 19: scaling experiments (population, dimension, OpenMP threads, CUDA evaluation modes). Measured data only.
from matplotlib.ticker import FuncFormatter
plain = FuncFormatter(lambda v, _: f"{v:g}")
SCALE_FUNC = "F19_Rastrigin"
SP = dict(
    smoke=dict(pops=[16, 64], dims=[5, 10], fix_dim=10, fix_pop=32, iters=10, runs=2, thr_pop=64, thr_dim=10, weak_pop=32,
               modes=[(32, 10)]),
    standard=dict(pops=[64, 256, 1024, 4096], dims=[10, 50, 100, 500], fix_dim=100, fix_pop=1024, iters=50, runs=5,
                  thr_pop=1024, thr_dim=100, weak_pop=512, modes=[(256, 100), (1024, 100), (256, 500), (4096, 100)]),
)
SP["final"] = dict(SP["standard"], runs=20)
S = SP[PROFILE]; SCALE_ITERS, SCALE_RUNS = S["iters"], S["runs"]
THREADS = sorted({t for t in [1, 2, 4, 8, 16, 32, 64] if t <= OMP_THREADS} | {OMP_THREADS})
sweep_skipped = []

def sweep(var, values):
    rows = []
    for val in values:
        pop, dim = (val, S["fix_dim"]) if var == "pop" else (S["fix_pop"], val)
        for v in IMPLS:
            if v == "cuda" and not fits_on_gpu(pop, dim):
                sweep_skipped.append((var, val, v)); continue
            df = run(v, func=SCALE_FUNC, dim=dim, pop=pop, iters=SCALE_ITERS, runs=SCALE_RUNS); df["sweep"] = val; rows.append(df)
        print(f"  {var} = {val} done")
    out = pd.concat(rows, ignore_index=True); out["seconds"] = out["time_ms"] / 1000.0; out["compute_seconds"] = out["compute_ms"] / 1000.0
    return out

def plot_sweep(df, xlabel, title, fname):
    g = df.groupby(["sweep", "version"])
    t, sd = g["seconds"].mean().unstack("version"), g["seconds"].std().unstack("version")
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for v in IMPLS:
        axes[0].errorbar(t.index, t[v], yerr=sd[v], marker="o", capsize=3, label=LABEL[v], color=COLORS[v])
    axes[0].set_xscale("log"); axes[0].set_yscale("log"); axes[0].xaxis.set_major_formatter(plain)
    axes[0].set_xlabel(xlabel); axes[0].set_ylabel("Mean end-to-end time per run (s)"); axes[0].set_title(title + ": runtime"); axes[0].grid(alpha=0.3, which="both"); axes[0].legend()
    for v in IMPLS[1:]:
        axes[1].plot(t.index, t["seq"] / t[v], marker="o", label=f"Sequential / {LABEL[v]}", color=COLORS[v])
    axes[1].axhline(1, ls="--", color="k"); axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].xaxis.set_major_formatter(plain); axes[1].yaxis.set_major_formatter(plain)
    axes[1].set_xlabel(xlabel); axes[1].set_ylabel("Speedup (> 1: parallel faster)"); axes[1].set_title(title + ": speedup"); axes[1].grid(alpha=0.3, which="both"); axes[1].legend()
    fig.suptitle(f"{SCALE_FUNC}, T = {SCALE_ITERS} iterations, {SCALE_RUNS} runs per point"); plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()
    return t

print("Population sweep"); pop_sweep = sweep("pop", S["pops"])
print("Dimension sweep");  dim_sweep = sweep("dim", S["dims"])
t_pop = plot_sweep(pop_sweep, f"Population size N (D = {S['fix_dim']})", "Population scaling", "fig_scaling_population.png")
t_dim = plot_sweep(dim_sweep, f"Dimension D (N = {S['fix_pop']})", "Dimension scaling", "fig_scaling_dimension.png")
print("Mean end-to-end seconds, population sweep:"); display(t_pop.round(5)); print("Mean end-to-end seconds, dimension sweep:"); display(t_dim.round(5))
pop_sweep.to_csv("heo_pop_sweep.csv", index=False); dim_sweep.to_csv("heo_dim_sweep.csv", index=False)

# ---- OpenMP strong scaling (fixed problem) and weak scaling (N proportional to threads)
base = run("seq", func=SCALE_FUNC, dim=S["thr_dim"], pop=S["thr_pop"], iters=SCALE_ITERS, runs=SCALE_RUNS)["time_ms"].mean() / 1000.0
strong = []
for th in THREADS:
    d = run("omp", func=SCALE_FUNC, dim=S["thr_dim"], pop=S["thr_pop"], iters=SCALE_ITERS, runs=SCALE_RUNS, threads=th)
    strong.append(dict(threads=th, seconds=d["time_ms"].mean() / 1000.0, std=d["time_ms"].std(ddof=1) / 1000.0))
strong = pd.DataFrame(strong); strong["speedup_vs_1thread"] = strong.seconds.iloc[0] / strong.seconds
strong["efficiency"] = strong.speedup_vs_1thread / strong.threads; strong["speedup_vs_sequential"] = base / strong.seconds
weak = []
for th in THREADS:
    d = run("omp", func=SCALE_FUNC, dim=S["thr_dim"], pop=S["weak_pop"] * th, iters=SCALE_ITERS, runs=SCALE_RUNS, threads=th)
    weak.append(dict(threads=th, pop=S["weak_pop"] * th, seconds=d["time_ms"].mean() / 1000.0))
weak = pd.DataFrame(weak); weak["weak_efficiency"] = weak.seconds.iloc[0] / weak.seconds
print(f"\nOpenMP strong scaling (N = {S['thr_pop']}, D = {S['thr_dim']}; sequential time {base:.4f} s)"); display(strong.round(4))
print(f"OpenMP weak scaling (N = {S['weak_pop']} x threads)"); display(weak.round(4))
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].plot(strong.threads, strong.speedup_vs_1thread, "o-", color=COLORS["omp"], label="measured"); ax[0].plot(strong.threads, strong.threads, "k--", label="ideal")
ax[0].set_xlabel("OpenMP threads"); ax[0].set_ylabel("Speedup vs 1 thread"); ax[0].set_title("Strong scaling: speedup"); ax[0].grid(alpha=0.3); ax[0].legend()
ax[1].plot(strong.threads, strong.efficiency, "o-", color=COLORS["omp"]); ax[1].axhline(1, ls="--", color="k"); ax[1].set_ylim(0, 1.15)
ax[1].set_xlabel("OpenMP threads"); ax[1].set_ylabel("Parallel efficiency"); ax[1].set_title("Strong scaling: efficiency"); ax[1].grid(alpha=0.3)
ax[2].plot(weak.threads, weak.weak_efficiency, "o-", color=COLORS["omp"]); ax[2].axhline(1, ls="--", color="k"); ax[2].set_ylim(0, 1.15)
ax[2].set_xlabel("OpenMP threads (N proportional to threads)"); ax[2].set_ylabel("T(1) / T(p)"); ax[2].set_title("Weak scaling efficiency"); ax[2].grid(alpha=0.3)
fig.suptitle(f"OpenMP thread scaling on {OMP_THREADS} visible CPU thread(s)"); plt.tight_layout(); plt.savefig("fig_scaling_threads.png", dpi=150); plt.show()
if OMP_THREADS < 4: print("Only", OMP_THREADS, "CPU thread(s) are visible, so the thread curves have very few points.")
strong.to_csv("heo_strong_scaling.csv", index=False); weak.to_csv("heo_weak_scaling.csv", index=False)

# ---- CUDA: one thread per hare vs several threads per hare for the evaluation (GPU only)
mode_study = None
if HAS_GPU:
    rows = []
    for pop, dim in S["modes"]:
        if not fits_on_gpu(pop, dim): sweep_skipped.append(("modes", (pop, dim), "cuda")); continue
        res = {m: run("cuda", func=SCALE_FUNC, dim=dim, pop=pop, iters=SCALE_ITERS, runs=SCALE_RUNS, env_extra={"HEO_CUDA_EVAL": m}) for m in ("thread", "terms")}
        rows.append(dict(pop=pop, dim=dim, thread_compute_ms=res["thread"].compute_ms.mean(), terms_compute_ms=res["terms"].compute_ms.mean(),
                         terms_speedup=res["thread"].compute_ms.mean() / res["terms"].compute_ms.mean(),
                         bit_identical=bool((res["thread"].best.values == res["terms"].best.values).all())))
    mode_study = pd.DataFrame(rows); print("\nCUDA evaluation mode: one thread per hare (thread) vs several (terms), compute-only time"); display(mode_study.round(3))
else:
    print("\nCUDA evaluation-mode study NOT executed: no usable GPU in this runtime.")
if sweep_skipped: print("Skipped (GPU memory):", sweep_skipped)
SCALE_META = dict(func=SCALE_FUNC, iters=SCALE_ITERS, runs=SCALE_RUNS, threads=THREADS, cuda_mode_study=mode_study is not None)


In [ ]:
# Cell 20: final reporting checklist. Generated from the actual run: every number below comes from variables
# produced by the cells above; anything that could not be executed is listed explicitly.
try:
    from IPython.display import Markdown, display as _show
except ImportError:                                   # plain Python: print the markdown instead
    Markdown, _show = (lambda s: s), print
def _g(name, default=None): return globals().get(name, default)
gm = lambda s: float(np.exp(np.log(s.dropna()).mean())) if len(s.dropna()) else float("nan")      # geometric mean
gpu = ENV.get("gpu") or {}
L = []
add = L.append
add(f"# HEO Sequential / OpenMP / CUDA: final report (profile `{_g('PROFILE')}`, finished {_g('RUN_META', {}).get('finished', 'n/a')})\n")
add("## 1. Hardware configuration")
add(f"* CPU: {ENV.get('cpu_model')} | visible CPU threads: {ENV.get('cpu_count')} | RAM: {ENV.get('ram_gb', 'n/a')} GB")
add(f"* GPU: {gpu.get('name', 'none used in this run')}" + (f" | {gpu.get('memory.total')} MiB | compute capability {gpu.get('compute_cap', 'n/a')} | driver {gpu.get('driver_version')}" if gpu else ""))
add("\n## 2. Software configuration")
add(f"* Python {ENV.get('python')} | {ENV.get('platform')}")
add(f"* Compiler: {ENV.get('gxx')} (OpenMP {ENV.get('openmp_macro')}) | CUDA toolkit: {ENV.get('nvcc') or 'not available'}")
add("* Build flags: CPU `-O3 -std=c++17 -ffp-contract=off` (+ `-fopenmp`); CUDA `-O3 -std=c++17 -arch=" + str(_g("CUDA_ARCH") or "n/a") + " --fmad=false`")
add("\n## 3. Algorithm parameters")
add("* Update (paper Eq. 4): `X' = clamp(X + c_t V + L (X+V) r)`, `V' = 0.95 V`; velocity-reduction coefficient 1, damping 0.95")
add("* Default coefficient `c_t = (T-t+1)/T` (documented interpretation); the paper's literal `Maxit/t` is available with `--scale literal` and was **not** used for the results below")
add("* L: Mantegna Levy step, beta = 1.5, unscaled (sigma_u = 0.6966); r ~ U(0,1); bounds handled by clamping; elitist merge-and-keep-best-N selection (parents win ties)")
add("* Initial positions and velocities uniform in the function bounds; counter-based RNG (splitmix64)")
add("\n## 4. Benchmark methodology")
M = _g("RUN_META", {})
if M:
    add(f"* Configurations (D, N, T): " + "; ".join(f"({c['dim']}, {c['pop']}, {c['iters']})" for c in M["configs"]))
    add(f"* Independent runs per configuration: {M['runs']} ({'meets' if M['runs'] >= 20 else 'BELOW'} the 20-run requirement) | shifted-optimum study: {M['shift_runs']} runs on one configuration")
    add(f"* Matched seeds: run k uses {M['base_seed']} + 1000003 k in every back-end | OpenMP threads: {M['omp_threads']} | back-ends: {', '.join(M['impls'])}")
    add(f"* Timing: end-to-end wall clock per run, and compute-only loop time (CUDA events on the GPU); untimed warm-up of identical size; total benchmark time {M['elapsed_s']} s")
add("\n## 5. Correctness validation")
add("* `--selftest` on every executable: RNG statistics, Levy sigma, benchmark optima, update rule, merge selection (see Section 3 output)")
par = _g("parity")
if par is not None and len(par):
    for _, r in par.iterrows():
        add(f"* {LABEL[r.version]} vs sequential over {int(r.runs)} matched-seed runs: {int(r.bit_identical)} bit-identical, max abs diff {r.max_abs:.3g}, max relative diff {r.max_rel:.3g}, runs above 1e-6: {int(r.runs_rel_gt_1e6)}")
add("\n## 6. Runtime comparison, speedup and efficiency (scalable functions, unshifted; geometric mean over functions)")
sp = _g("speedup")
if sp is not None:
    d = sp[(sp["shift"] == 0) & sp.func.isin(SCALABLE)]
    tab = d.groupby(["cfg", "version"]).agg(end_to_end=("speedup_end_to_end", gm), compute_only=("speedup_compute", gm), efficiency=("efficiency", gm), overhead=("overhead_frac", "mean")).round(3)
    try: add(tab.to_markdown())
    except ImportError: add("```\n" + tab.to_string() + "\n```")
add("\n(efficiency = speedup / OpenMP threads; not defined for CUDA)")
add("\n## 7. Statistical analysis")
for nm, w in (("final fitness", _g("w_fit")), ("runtime", _g("w_time"))):
    if w is not None:
        t = w.dropna(subset=["p_value"]); add(f"* Wilcoxon signed-rank, {nm}: {len(w)} comparisons, {len(w) - len(t)} identical or untestable, {int((t.p_value < 0.05).sum())}/{len(t)} testable with p < 0.05 (uncorrected)")
add(f"* Friedman, fitness: {friedman('best')} | runtime: {friedman('seconds')}")
add("\n## 8. Not executed / caveats about this run")
if not HAS_GPU: add("* **CUDA was not built or run** (no usable GPU/nvcc): no GPU timing, GPU parity, Friedman test or CUDA evaluation-mode study exists for this run")
if M and M["runs"] < 20: add(f"* **Profile `{M['profile']}` used {M['runs']} runs**: not a final experiment (needs >= 20); Wilcoxon tests need at least 6 runs")
if _g("sweep_skipped"): add(f"* Points skipped for GPU memory: {sweep_skipped}")
add("* The paper's quality claims (29 competitors, CEC 2015 shifted/rotated functions, engineering problems) are **not** reproduced here")
add("\n## 9. Limitations")
add("* Several equations of the paper are ambiguous; the choices (A1-A9 in `heo_common.h`) affect fitness but not the parallel-performance conclusions")
add("* The update term scales with the position itself, which favours optima at the origin (see the shifted-optimum figure)")
add("* Timings reflect this hardware and these sizes; one-time allocation is excluded by the warm-up; the sort and gather keep part of the work memory-bound")
add("* CUDA agrees with the CPU only up to device math-library rounding; a changed selection order can, rarely, change a trajectory")
add("* Wilcoxon p-values are not corrected for multiple comparisons")
add("\n## 10. Future improvements")
add("* CUDA Graphs or a persistent kernel to remove per-iteration launch latency; a device-side bitonic sort for small N")
add("* Float (`-DUSE_FLOAT`) and tensor-friendly layouts; multi-GPU and CPU+GPU hybrid populations")
add("* CEC 2015 functions with the official shift/rotation data, constrained engineering problems, adaptive parameter tuning")
REPORT = "\n".join(L)
open("heo_report.md", "w").write(REPORT)
_show(Markdown(REPORT))
print("\nReport also saved to heo_report.md")
